# Подготовка вводных данных

### --- Pip & libs

In [ ]:
!pip install pod5
!pip install edlib -q
!pip install lz4

In [ ]:
import os
import sys
import time
import pod5
import json
import edlib
import shutil
import random
import zipfile
import hashlib
import warnings
import importlib
import lz4.frame
import subprocess

import numpy as np
import pandas as pd
import zstandard as zstd
import matplotlib.pyplot as plt


from pathlib import Path
from datetime import datetime
from google.colab import drive
from google.colab import files
from collections import Counter
from collections import defaultdict

drive.mount('/content/drive')

In [ ]:
os.environ["PATH"] += ":/content/dorado-0.9.1-linux-x64/bin"
!dorado download --model dna_r10.4.1_e8.2_400bps_hac@v4.3.0
!dorado download --model dna_r9.4.1_e8_hac@v3.3

In [ ]:
# Скачиваем Dorado для Linux
!wget https://cdn.oxfordnanoportal.com/software/analysis/dorado-0.9.1-linux-x64.tar.gz
!tar -xzf dorado-0.9.1-linux-x64.tar.gz
!/content/dorado-0.9.1-linux-x64/bin/dorado --version

!dorado --version

In [ ]:
!dorado download --model dna_r10.4.1_e8.2_400bps_hac@v4.3.0
!dorado download --model dna_r9.4.1_e8_hac@v3.3

In [ ]:
!find /content/drive -name "*guppy*" 2>/dev/null

In [ ]:
!cp /content/drive/MyDrive/ont-guppy_6.5.7_linux64.tar.gz /content/
!tar -xzf /content/ont-guppy_6.5.7_linux64.tar.gz -C /content/
!ls /content/ont-guppy/bin/

In [ ]:
!/content/ont-guppy/bin/guppy_basecaller --version

In [ ]:
!/content/ont-guppy/bin/guppy_basecaller --print_workflows | grep "FLO-MIN112"

In [ ]:
!/content/ont-guppy/bin/guppy_basecaller --help 2>&1 | grep -i "pod5\|fast5"

### --- Распаковка архива с данными

In [ ]:
# Архив на Google Drive
zip_path = Path("/content/drive/MyDrive/POD5.zip")

# Папка в локальной среде Colab
out_dir = Path("/content/POD5")

# Если нужно полностью очистить старую распаковку
CLEAR_OUTPUT = False

if not zip_path.exists():
    raise FileNotFoundError(f"Архив не найден: {zip_path}")

if CLEAR_OUTPUT and out_dir.exists():
    shutil.rmtree(out_dir)

out_dir.mkdir(parents=True, exist_ok=True)

print(f"Распаковка архива:\n{zip_path}\n→ {out_dir}")

with zipfile.ZipFile(zip_path, "r") as zf:
    bad_file = zf.testzip()
    if bad_file is not None:
        raise RuntimeError(f"Архив повреждён. Проблемный файл: {bad_file}")
    zf.extractall(out_dir)

pod5_files = list(out_dir.rglob("*.pod5"))

print("Готово.")
print(f"Папка распаковки: {out_dir}")
print(f"Найдено POD5-файлов: {len(pod5_files)}")

for p in pod5_files[:10]:
    print(p)

In [ ]:
# Папка с данными
BASE_DIR = Path("/content/POD5")

# Анализ данных

## --- Анализ таблиц POD5

### --- --- Глобальные переменные

In [ ]:
BASE_DIR = Path("/content/POD5")
OUT_DIR = Path("/content/pod5_table_structure_report")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Кол-во файлов
N_FILES = 5

# Кол-во ридов
MAX_READS_PER_FILE = None

# Сколько значений показывать в примерах.
N_SAMPLE_VALUES = 5

# Сколько строк показывать в больших отчётах.
N_SHOW_ROWS = 80

### --- --- Реализация

In [ ]:
def safe_str(x):
    try:
        return str(x)
    except Exception:
        return repr(x)


def enum_name(x):
    if x is None:
        return None
    if hasattr(x, "name"):
        try:
            return x.name
        except Exception:
            pass
    return safe_str(x)


def scalarize(x):
    if x is None:
        return None

    if isinstance(x, (str, int, float, bool)):
        return x

    if isinstance(x, np.generic):
        return x.item()

    if isinstance(x, Path):
        return str(x)

    if hasattr(x, "isoformat"):
        try:
            return x.isoformat()
        except Exception:
            pass

    if isinstance(x, (list, tuple)):
        if len(x) <= 10:
            return json.dumps([scalarize(v) for v in x], ensure_ascii=False)
        return f"<{type(x).__name__}: len={len(x)}>"

    if isinstance(x, dict):
        try:
            return json.dumps(
                {str(k): scalarize(v) for k, v in x.items()},
                ensure_ascii=False,
                sort_keys=True
            )
        except Exception:
            return safe_str(x)

    if hasattr(x, "name"):
        try:
            return x.name
        except Exception:
            pass

    return safe_str(x)


def get_attr(obj, name, default=None):
    if obj is None:
        return default
    try:
        return getattr(obj, name)
    except Exception:
        return default


def get_nested(obj, *names, default=None):
    cur = obj
    for name in names:
        cur = get_attr(cur, name, default=None)
        if cur is None:
            return default
    return cur


def flatten_obj(obj, prefix="", max_depth=3):
    """
    Аккуратно разворачивает объект POD5 в плоский словарь.
    В первую очередь нужно для RunInfo и его tracking_id/context_tags.
    """
    out = {}

    if obj is None:
        return out

    if max_depth < 0:
        key = prefix.rstrip("__") or "value"
        return {key: scalarize(obj)}

    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}{k}"
            if isinstance(v, dict):
                out.update(flatten_obj(v, prefix=f"{key}__", max_depth=max_depth-1))
            else:
                out[key] = scalarize(v)
        return out

    # Если объект выглядит как простое значение, не надо разворачивать его внутренности.
    if isinstance(obj, (str, int, float, bool, np.generic)):
        key = prefix.rstrip("__") or "value"
        return {key: scalarize(obj)}

    # dataclass / обычный объект / slotted объект
    items = []

    if hasattr(obj, "__dict__"):
        try:
            items = list(vars(obj).items())
        except Exception:
            items = []

    if not items:
        for name in dir(obj):
            if name.startswith("_"):
                continue
            try:
                value = getattr(obj, name)
            except Exception:
                continue
            if callable(value):
                continue
            items.append((name, value))

    for name, value in items:
        if name.startswith("_") or callable(value):
            continue

        key = f"{prefix}{name}"

        if isinstance(value, dict):
            out.update(flatten_obj(value, prefix=f"{key}__", max_depth=max_depth-1))
        elif hasattr(value, "__dict__") and max_depth > 0 and not isinstance(value, (str, int, float, bool)):
            out.update(flatten_obj(value, prefix=f"{key}__", max_depth=max_depth-1))
        else:
            out[key] = scalarize(value)

    return out


def stable_hash(d):
    text = json.dumps(d, ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]


def entropy_bits(series):
    s = series.dropna().astype(str)
    if len(s) == 0:
        return 0.0
    counts = s.value_counts()
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())


def sample_values(series, n=N_SAMPLE_VALUES):
    vals = series.dropna().astype(str).unique()[:n]
    return " | ".join(vals)


def format_df(df, max_rows=N_SHOW_ROWS, max_colwidth=90):
    if df is None or len(df) == 0:
        return "(empty)"
    d = df.head(max_rows).copy()
    for col in d.columns:
        d[col] = d[col].map(lambda x: safe_str(x)[:max_colwidth])
    return d.to_string(index=False)


def profile_table(df, table_name):
    rows = len(df)
    result = []

    for col in df.columns:
        s = df[col]
        non_null = int(s.notna().sum())
        unique_values = int(s.nunique(dropna=True))
        unique_ratio = unique_values / rows if rows > 0 else 0.0
        ent = entropy_bits(s)

        if unique_values == 0:
            role = "all_null"
        elif unique_values == 1:
            role = "constant"
        elif rows > 0 and unique_values == rows and non_null == rows:
            role = "unique_candidate_key"
        elif unique_ratio <= 0.01:
            role = "low_cardinality"
        elif unique_ratio <= 0.10:
            role = "medium_cardinality"
        else:
            role = "high_cardinality"

        result.append({
            "table": table_name,
            "column": col,
            "rows": rows,
            "non_null": non_null,
            "nulls": rows - non_null,
            "dtype": str(s.dtype),
            "unique_values": unique_values,
            "unique_ratio": round(unique_ratio, 6),
            "entropy_bits": round(ent, 4),
            "role": role,
            "sample_values": sample_values(s),
        })

    return pd.DataFrame(result)


def fd_check(df, table_name, determinant_cols, dependent_cols=None):
    """
    Проверка функциональной зависимости:
    determinant_cols -> dependent_col

    holds=True означает, что для каждого значения determinant_cols
    dependent_col имеет не более одного значения.
    """
    if dependent_cols is None:
        dependent_cols = [c for c in df.columns if c not in determinant_cols]

    rows = []

    missing = [c for c in determinant_cols if c not in df.columns]
    if missing:
        return pd.DataFrame([{
            "table": table_name,
            "determinant": ", ".join(determinant_cols),
            "dependent": None,
            "holds": False,
            "max_unique_per_determinant": None,
            "note": f"Missing determinant columns: {missing}"
        }])

    for dep in dependent_cols:
        if dep not in df.columns:
            continue

        try:
            grouped = df.groupby(determinant_cols, dropna=False)[dep].nunique(dropna=False)
            max_unique = int(grouped.max()) if len(grouped) else 0
            holds = max_unique <= 1
        except Exception as e:
            max_unique = None
            holds = False
            rows.append({
                "table": table_name,
                "determinant": ", ".join(determinant_cols),
                "dependent": dep,
                "holds": holds,
                "max_unique_per_determinant": max_unique,
                "note": f"Error: {repr(e)}"
            })
            continue

        rows.append({
            "table": table_name,
            "determinant": ", ".join(determinant_cols),
            "dependent": dep,
            "holds": holds,
            "max_unique_per_determinant": max_unique,
            "note": "FD holds" if holds else "FD violated"
        })

    return pd.DataFrame(rows)


def normalize_name(x):
    return str(x).lower().replace("_", "").replace("-", "").replace(" ", "")


In [ ]:
# 3. Locate POD5 files
pod5_files_all = sorted(BASE_DIR.rglob("*.pod5"))

if N_FILES is None:
    files = pod5_files_all
else:
    files = pod5_files_all[:N_FILES]

print("POD5 analysis started")
print("BASE_DIR:", BASE_DIR)
print("OUT_DIR:", OUT_DIR)
print("Total POD5 files found:", len(pod5_files_all))
print("Files selected:", len(files))
print("MAX_READS_PER_FILE:", MAX_READS_PER_FILE)

if len(files) == 0:
    raise FileNotFoundError(
        f"No .pod5 files found in {BASE_DIR}. "
        "Check that POD5.zip was extracted into /content/POD5."
    )

In [ ]:
# 4. Extract logical tables

reads_rows = []
signal_rows = []
runinfo_by_id = {}
link_rows = []
errors = []

for file_idx, pod5_path in enumerate(files, start=1):
    print(f"[{file_idx}/{len(files)}] Reading: {pod5_path}")

    try:
        with pod5.Reader(str(pod5_path)) as reader:
            for read_idx, read in enumerate(reader.reads()):
                if MAX_READS_PER_FILE is not None and read_idx >= MAX_READS_PER_FILE:
                    break

                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")

                        read_id = scalarize(get_attr(read, "read_id"))

                        run_info = get_attr(read, "run_info")
                        run_info_dict = flatten_obj(run_info, max_depth=3)

                        acquisition_id = run_info_dict.get("acquisition_id")
                        if acquisition_id not in [None, "None", ""]:
                            run_info_id = str(acquisition_id)
                        else:
                            run_info_id = stable_hash(run_info_dict)

                        if run_info_id not in runinfo_by_id:
                            runinfo_row = {
                                "run_info_id": run_info_id,
                                "source_file_first_seen": str(pod5_path),
                            }
                            runinfo_row.update(run_info_dict)
                            runinfo_by_id[run_info_id] = runinfo_row

                        pore = get_attr(read, "pore")
                        calibration = get_attr(read, "calibration")
                        end_reason = get_attr(read, "end_reason")

                        tracked_scaling = get_attr(read, "tracked_scaling")
                        predicted_scaling = get_attr(read, "predicted_scaling")

                        sig = np.asarray(get_attr(read, "signal"))

                        if sig.size > 0:
                            sig_min = int(sig.min())
                            sig_max = int(sig.max())
                            sig_first = int(sig[0])
                            sig_last = int(sig[-1])
                            sig_mean = float(sig.mean())
                            sig_std = float(sig.std())
                        else:
                            sig_min = None
                            sig_max = None
                            sig_first = None
                            sig_last = None
                            sig_mean = None
                            sig_std = None

                        reads_rows.append({
                            "source_file": str(pod5_path),
                            "read_id": str(read_id),
                            "run_info_id": run_info_id,

                            "read_number": scalarize(get_attr(read, "read_number")),
                            "start_sample": scalarize(get_attr(read, "start_sample")),
                            "num_minknow_events": scalarize(get_attr(read, "num_minknow_events")),
                            "median_before": scalarize(get_attr(read, "median_before")),

                            "channel": scalarize(get_nested(read, "pore", "channel")),
                            "well": scalarize(get_nested(read, "pore", "well")),
                            "pore_type": enum_name(get_nested(read, "pore", "pore_type")),

                            "calibration_offset": scalarize(get_nested(read, "calibration", "offset")),
                            "calibration_scale": scalarize(get_nested(read, "calibration", "scale")),

                            "end_reason": enum_name(end_reason),
                            "end_reason_forced": scalarize(get_attr(end_reason, "forced")),

                            "tracked_scaling_scale": scalarize(get_attr(tracked_scaling, "scale")),
                            "tracked_scaling_shift": scalarize(get_attr(tracked_scaling, "shift")),
                            "predicted_scaling_scale": scalarize(get_attr(predicted_scaling, "scale")),
                            "predicted_scaling_shift": scalarize(get_attr(predicted_scaling, "shift")),
                        })

                        signal_rows.append({
                            "source_file": str(pod5_path),
                            "read_id": str(read_id),
                            "signal_dtype": str(sig.dtype),
                            "signal_len": int(sig.size),
                            "signal_nbytes": int(sig.nbytes),
                            "signal_min": sig_min,
                            "signal_max": sig_max,
                            "signal_range": None if sig_min is None or sig_max is None else sig_max - sig_min,
                            "signal_first": sig_first,
                            "signal_last": sig_last,
                            "signal_mean": sig_mean,
                            "signal_std": sig_std,
                        })

                        link_rows.append({
                            "source_file": str(pod5_path),
                            "read_id": str(read_id),
                            "signal_link": str(read_id),
                            "run_info_id": run_info_id,
                        })

                except Exception as e:
                    errors.append({
                        "source_file": str(pod5_path),
                        "read_index": read_idx,
                        "error": repr(e),
                    })

    except Exception as e:
        errors.append({
            "source_file": str(pod5_path),
            "read_index": None,
            "error": repr(e),
        })


reads_df = pd.DataFrame(reads_rows)
signal_df = pd.DataFrame(signal_rows)
runinfo_df = pd.DataFrame(list(runinfo_by_id.values()))
links_df = pd.DataFrame(link_rows)
errors_df = pd.DataFrame(errors)

In [ ]:
# 5. Save raw extracted tables

reads_df.to_csv(OUT_DIR / "reads_table.csv", index=False)
signal_df.to_csv(OUT_DIR / "signal_summary_table.csv", index=False)
runinfo_df.to_csv(OUT_DIR / "runinfo_table.csv", index=False)
links_df.to_csv(OUT_DIR / "links_table.csv", index=False)
errors_df.to_csv(OUT_DIR / "errors.csv", index=False)

In [ ]:
# 6. Build profiles

profile_parts = []

if not reads_df.empty:
    profile_parts.append(profile_table(reads_df, "Reads"))

if not signal_df.empty:
    profile_parts.append(profile_table(signal_df, "SignalSummary"))

if not runinfo_df.empty:
    profile_parts.append(profile_table(runinfo_df, "RunInfo"))

if not links_df.empty:
    profile_parts.append(profile_table(links_df, "Links"))

profiles_df = pd.concat(profile_parts, ignore_index=True) if profile_parts else pd.DataFrame()
if not profiles_df.empty:
    profiles_df = profiles_df.sort_values(["table", "unique_ratio", "entropy_bits", "column"])

profiles_df.to_csv(OUT_DIR / "column_profiles.csv", index=False)

In [ ]:
# 7. Relationship checks

relationship_rows = []

def add_relationship(name, value):
    relationship_rows.append({
        "relationship_check": name,
        "value": value,
    })

add_relationship("POD5 files found total", len(pod5_files_all))
add_relationship("POD5 files processed", len(files))
add_relationship("Reads rows", len(reads_df))
add_relationship("SignalSummary rows", len(signal_df))
add_relationship("RunInfo rows", len(runinfo_df))
add_relationship("Links rows", len(links_df))
add_relationship("Errors rows", len(errors_df))

if not reads_df.empty:
    add_relationship("Reads unique read_id", reads_df["read_id"].nunique())
    add_relationship("Reads read_id is unique", reads_df["read_id"].nunique() == len(reads_df))
    add_relationship("Reads unique run_info_id", reads_df["run_info_id"].nunique())

if not signal_df.empty:
    add_relationship("SignalSummary unique read_id", signal_df["read_id"].nunique())
    add_relationship("SignalSummary read_id is unique", signal_df["read_id"].nunique() == len(signal_df))

if not runinfo_df.empty:
    add_relationship("RunInfo unique run_info_id", runinfo_df["run_info_id"].nunique())
    add_relationship("RunInfo run_info_id is unique", runinfo_df["run_info_id"].nunique() == len(runinfo_df))

if not reads_df.empty and not signal_df.empty:
    reads_ids = set(reads_df["read_id"])
    signal_ids = set(signal_df["read_id"])

    add_relationship("Reads without SignalSummary", len(reads_ids - signal_ids))
    add_relationship("SignalSummary without Reads", len(signal_ids - reads_ids))
    add_relationship(
        "Reads to SignalSummary relation",
        "1:1 by read_id" if reads_ids == signal_ids and len(reads_df) == len(signal_df) else "not perfect 1:1"
    )

if not reads_df.empty and not runinfo_df.empty:
    reads_run_ids = set(reads_df["run_info_id"])
    runinfo_ids = set(runinfo_df["run_info_id"])

    add_relationship("Reads run_info_id missing in RunInfo", len(reads_run_ids - runinfo_ids))
    add_relationship("RunInfo ids not used by Reads", len(runinfo_ids - reads_run_ids))

    reads_per_run = reads_df.groupby("run_info_id")["read_id"].count()
    add_relationship("Reads per RunInfo min", int(reads_per_run.min()))
    add_relationship("Reads per RunInfo max", int(reads_per_run.max()))
    add_relationship("Reads per RunInfo mean", round(float(reads_per_run.mean()), 4))

relationships_df = pd.DataFrame(relationship_rows)
relationships_df.to_csv(OUT_DIR / "relationships_report.csv", index=False)

In [ ]:
# 8. Functional dependency checks

fd_reports = []

if not reads_df.empty and "read_id" in reads_df.columns:
    fd_reports.append(fd_check(
        reads_df,
        "Reads",
        ["read_id"],
        [c for c in reads_df.columns if c != "read_id"]
    ))

if not signal_df.empty and "read_id" in signal_df.columns:
    fd_reports.append(fd_check(
        signal_df,
        "SignalSummary",
        ["read_id"],
        [c for c in signal_df.columns if c != "read_id"]
    ))

if not runinfo_df.empty and "run_info_id" in runinfo_df.columns:
    fd_reports.append(fd_check(
        runinfo_df,
        "RunInfo",
        ["run_info_id"],
        [c for c in runinfo_df.columns if c != "run_info_id"]
    ))

if not reads_df.empty and "run_info_id" in reads_df.columns:
    fd_reports.append(fd_check(
        reads_df,
        "Reads",
        ["run_info_id"],
        [c for c in reads_df.columns if c not in ["run_info_id", "read_id"]]
    ))

fd_df = pd.concat(fd_reports, ignore_index=True) if fd_reports else pd.DataFrame()
fd_df.to_csv(OUT_DIR / "functional_dependencies_report.csv", index=False)

if not fd_df.empty:
    reads_by_run_fd_df = fd_df[
        (fd_df["table"] == "Reads") &
        (fd_df["determinant"] == "run_info_id")
    ].copy()
else:
    reads_by_run_fd_df = pd.DataFrame()

reads_fields_constant_within_runinfo_df = pd.DataFrame()
if not reads_by_run_fd_df.empty:
    reads_fields_constant_within_runinfo_df = reads_by_run_fd_df[
        reads_by_run_fd_df["holds"] == True
    ].copy()
    reads_fields_constant_within_runinfo_df.to_csv(
        OUT_DIR / "reads_fields_constant_within_runinfo.csv",
        index=False
    )

In [ ]:
# 9. RunInfo tracking_id duplication checks

dup_rows = []

if not runinfo_df.empty:
    tracking_cols = [c for c in runinfo_df.columns if c.startswith("tracking_id__")]
    other_cols = [c for c in runinfo_df.columns if not c.startswith("tracking_id__")]

    for tc in tracking_cols:
        short_tc = tc.replace("tracking_id__", "")
        norm_tc = normalize_name(short_tc)

        for oc in other_cols:
            norm_oc = normalize_name(oc)
            same_name = norm_tc == norm_oc

            try:
                left = runinfo_df[tc].astype(str).fillna("<NA>").tolist()
                right = runinfo_df[oc].astype(str).fillna("<NA>").tolist()
                same_values = left == right
            except Exception:
                same_values = False

            if same_name or same_values:
                dup_rows.append({
                    "tracking_column": tc,
                    "runinfo_column": oc,
                    "same_normalized_name": same_name,
                    "same_values": same_values,
                })

tracking_duplicates_df = pd.DataFrame(dup_rows)
tracking_duplicates_df.to_csv(OUT_DIR / "runinfo_tracking_duplicates.csv", index=False)


In [ ]:
# 10. Value counts for important categorical fields

value_count_rows = []

def add_value_counts(df, table, col, top_n=10):
    if df.empty or col not in df.columns:
        return
    vc = df[col].astype(str).fillna("<NA>").value_counts(dropna=False).head(top_n)
    total = len(df)
    for value, count in vc.items():
        value_count_rows.append({
            "table": table,
            "column": col,
            "value": value,
            "count": int(count),
            "share": round(float(count / total), 6) if total else None,
        })

for col in [
    "source_file",
    "run_info_id",
    "channel",
    "well",
    "pore_type",
    "end_reason",
    "end_reason_forced",
    "calibration_offset",
    "calibration_scale",
]:
    add_value_counts(reads_df, "Reads", col, top_n=12)

for col in [
    "signal_dtype",
    "signal_len",
    "signal_min",
    "signal_max",
    "signal_range",
]:
    add_value_counts(signal_df, "SignalSummary", col, top_n=12)

value_counts_df = pd.DataFrame(value_count_rows)
value_counts_df.to_csv(OUT_DIR / "selected_value_counts.csv", index=False)

In [ ]:
# 11. Compact interpretation helpers

def select_profile(role=None, table=None):
    if profiles_df.empty:
        return pd.DataFrame()
    d = profiles_df.copy()
    if role is not None:
        d = d[d["role"] == role]
    if table is not None:
        d = d[d["table"] == table]
    keep = [
        "table", "column", "rows", "non_null", "unique_values",
        "unique_ratio", "entropy_bits", "role", "sample_values"
    ]
    return d[keep].copy()


constants_df = select_profile(role="constant")
low_cardinality_df = select_profile(role="low_cardinality")
candidate_keys_df = select_profile(role="unique_candidate_key")
all_null_df = select_profile(role="all_null")

runinfo_field_summary_rows = []
if not runinfo_df.empty:
    for col in runinfo_df.columns:
        s = runinfo_df[col]
        nunique = s.nunique(dropna=True)
        if nunique <= 3:
            vals = " | ".join(s.dropna().astype(str).unique()[:3])
        else:
            vals = f"{nunique} unique values"
        runinfo_field_summary_rows.append({
            "column": col,
            "unique_values": int(nunique),
            "values_or_note": vals[:160]
        })

runinfo_field_summary_df = pd.DataFrame(runinfo_field_summary_rows)
runinfo_field_summary_df.to_csv(OUT_DIR / "runinfo_field_summary.csv", index=False)

### --- --- Результат

In [ ]:
print("TABLE SHAPES")
print(f"Reads: {reads_df.shape}")
print(f"SignalSummary: {signal_df.shape}")
print(f"RunInfo: {runinfo_df.shape}")
print(f"Errors: {errors_df.shape}")

In [ ]:
print("READS COLUMNS")
print(", ".join(reads_df.columns))
print("\nSIGNAL COLUMNS")
print(", ".join(signal_df.columns))
print("\nRUNINFO COLUMNS")
print(", ".join(runinfo_df.columns))

In [ ]:
print("CANDIDATE KEYS")
print(format_df(candidate_keys_df))
print("\nCONSTANT COLUMNS")
print(format_df(constants_df))
print("\nALL-NULL COLUMNS")
print(format_df(all_null_df))

In [ ]:
print("FIELDS DETERMINED BY run_info_id")
print(format_df(reads_fields_constant_within_runinfo_df))

print("\nFIELDS NOT DETERMINED BY run_info_id")
reads_not_by_run_df = reads_by_run_fd_df[reads_by_run_fd_df["holds"] == False].copy()
print(format_df(reads_not_by_run_df[[
    "table", "determinant", "dependent", "holds", "max_unique_per_determinant", "note"
]]))

In [ ]:


print("RUNINFO FIELD SUMMARY")
print(format_df(runinfo_field_summary_df))
print("\nTRACKING_ID DUPLICATES")
print(format_df(tracking_duplicates_df))

In [ ]:
print("SELECTED VALUE COUNTS")
print(format_df(value_counts_df, max_rows=120))

## --- Анализ метаданных

In [ ]:
DATASETS = {
    "Bacterial R10.4": Path("/content/POD5/Bacterial R10.4 data"),
    "Plasmid R10.4.1": Path("/content/POD5/plasmid_2025.04"),
    "E. coli R9":      Path("/content/POD5/R9 data of E. coli (ERR9127551)"),
}

SKIP = {"barcode02_r0barcode02b48_0.pod5"} # битый

for name, d in DATASETS.items():
    f = next(f for f in d.glob("*.pod5") if f.name not in SKIP)
    with pod5.Reader(f) as r:
        read = next(r.reads())
        ri = read.run_info
        print(f"{name}: adc_min={ri.adc_min}, adc_max={ri.adc_max}")

### Анализ AUC

In [ ]:
DATASETS = {
    "Bacterial R10.4": Path("/content/POD5/Bacterial R10.4 data"),
    "Plasmid R10.4.1": Path("/content/POD5/plasmid_2025.04"),
    "E. coli R9":      Path("/content/POD5/R9 data of E. coli (ERR9127551)"),
}
SKIP = {"barcode02_r0barcode02b48_0.pod5"}
N_RANDOM_READS = 5   # ридов для индивидуальных графиков
N_FILES_AVG = 2      # файлов для усреднённого распределения
BINS = np.arange(-4096, 4096, 16)  # бины по 16 уровней АЦП

colors = ["blue", "red", "green"]
stats = {}

In [ ]:
PLOT_DIR = Path("/content/adc_plots")
PLOT_DIR.mkdir(exist_ok=True)

# Сбор ридов
reads_by_dataset = {}
for name, d in DATASETS.items():
    files = [f for f in d.glob("*.pod5") if f.name not in SKIP]
    collected = []
    for f in files:
        with pod5.Reader(f) as r:
            for read in r.reads():
                collected.append(read.signal.copy())
                if len(collected) >= 50:
                    break
        if len(collected) >= 50:
            break
    reads_by_dataset[name] = random.sample(collected, min(N_RANDOM_READS, len(collected)))
    print(f"  {name}: собрано {len(collected)} ридов, выбрано {len(reads_by_dataset[name])}")

# Отдельный график для каждого рида
for (name, reads), color in zip(reads_by_dataset.items(), colors):
    safe_name = name.replace(" ", "_").replace(".", "_")
    for i, sig in enumerate(reads):
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.hist(sig, bins=BINS, color=color, alpha=0.8)
        ax.set_title(f"{name} — рид {i+1} (n={len(sig)} отсчётов)")
        ax.set_xlabel("Значение АЦП")
        ax.set_ylabel("Число отсчётов")
        plt.tight_layout()
        fname = PLOT_DIR / f"{safe_name}_read_{i+1}.png"
        plt.savefig(fname, dpi=150)
        plt.close()
        print(f"  ✓ {fname.name}")

print(f"\nВсе графики сохранены в {PLOT_DIR}")

In [ ]:
# График 2: усреднённое распределение по всем ридам
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Усреднённое распределение АЦП по ридам", fontsize=14)

for ax, (name, d), color in zip(axes, DATASETS.items(), colors):
    files = [f for f in d.glob("*.pod5") if f.name not in SKIP][:N_FILES_AVG]
    hist_sum = np.zeros(len(BINS) - 1)
    n_reads = 0

    for f in files:
        with pod5.Reader(f) as r:
            for read in r.reads():
                h, _ = np.histogram(read.signal, bins=BINS)
                hist_sum += h / len(read.signal)  # нормируем на длину рида
                n_reads += 1

    hist_avg = hist_sum / n_reads
    bin_centers = (BINS[:-1] + BINS[1:]) / 2

    ax.bar(bin_centers, hist_avg, width=16, color=color, alpha=0.8)
    ax.set_title(f"{name}\n({n_reads} ридов)")
    ax.set_xlabel("Значение АЦП")
    ax.set_ylabel("Средняя доля отсчётов")
    ax.axvline(0, color='black', linewidth=0.5, linestyle='--')

plt.tight_layout()
plt.savefig("/content/adc_averaged_distribution.png", dpi=150)
plt.show()
print("График 2 сохранён")

In [ ]:
N_FILES = None

stats = {}
for name, d in DATASETS.items():
    print(f"Читаем {name}...")
    files = [f for f in d.glob("*.pod5") if f.name not in SKIP]
    if N_FILES:
        files = files[:N_FILES]
    sig_mins, sig_maxs, sig_ranges = [], [], []
    for f in files:
        with pod5.Reader(f) as r:
            for read in r.reads():
                s = read.signal
                sig_mins.append(int(s.min()))
                sig_maxs.append(int(s.max()))
                sig_ranges.append(int(s.max()) - int(s.min()))
    stats[name] = {
        "min": np.array(sig_mins),
        "max": np.array(sig_maxs),
        "range": np.array(sig_ranges),
    }
    print(f"  ридов: {len(sig_mins)}, медиана range: {np.median(sig_ranges):.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for (name, data), color in zip(stats.items(), colors):
    ax.hist(data["min"], bins=60, alpha=0.6, label=name, color=color)
ax.set_title("Распределение signal_min по ридам")
ax.set_xlabel("Минимальное значение АЦП в риде")
ax.set_ylabel("Число ридов")
ax.legend()
plt.tight_layout()
plt.savefig("/content/signal_min.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for (name, data), color in zip(stats.items(), colors):
    ax.hist(data["max"], bins=60, alpha=0.6, label=name, color=color)
ax.set_title("Распределение signal_max по ридам")
ax.set_xlabel("Максимальное значение АЦП в риде")
ax.set_ylabel("Число ридов")
ax.legend()
plt.tight_layout()
plt.savefig("/content/signal_max.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for (name, data), color in zip(stats.items(), colors):
    ax.hist(data["range"], bins=60, alpha=0.6, label=name, color=color)
ax.set_title("Распределение signal_range по ридам (max − min)")
ax.set_xlabel("Диапазон значений АЦП в риде")
ax.set_ylabel("Число ридов")
ax.axvline(512, color='black', linestyle='--', linewidth=0.8, label='9 бит')
ax.axvline(1024, color='gray', linestyle='--', linewidth=0.8, label='10 бит')
ax.legend()
plt.tight_layout()
plt.savefig("/content/signal_range.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.suptitle("Распределение signal_min по ридам")

for ax, (name, data), color in zip(axes, stats.items(), colors):
    ax.hist(data["min"], bins=60, color=color, alpha=0.8)
    ax.set_title(name)
    ax.set_xlabel("Минимальное значение АЦП")
    ax.set_ylabel("Число ридов")

plt.tight_layout()
plt.savefig("/content/signal_min_separate.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.suptitle("Распределение signal_max по ридам")

for ax, (name, data), color in zip(axes, stats.items(), colors):
    ax.hist(data["max"], bins=60, color=color, alpha=0.8)
    ax.set_title(name)
    ax.set_xlabel("Максимальное значение АЦП")
    ax.set_ylabel("Число ридов")

plt.tight_layout()
plt.savefig("/content/signal_max_separate.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.suptitle("Распределение signal_range по ридам (max − min)")

for ax, (name, data), color in zip(axes, stats.items(), colors):
    ax.hist(data["range"], bins=60, color=color, alpha=0.8)
    ax.set_title(name)
    ax.set_xlabel("Диапазон АЦП")
    ax.set_ylabel("Число ридов")
    ax.axvline(512, color='black', linestyle='--', linewidth=0.8, label='9 бит')
    ax.axvline(1024, color='gray', linestyle='--', linewidth=0.8, label='10 бит')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("/content/signal_range_separate.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.suptitle("Распределение эффективных бит по ридам (log₂(max−min))", fontsize=13)

for ax, (name, d), color in zip(axes, DATASETS.items(), colors):
    files = [f for f in d.glob("*.pod5") if f.name not in SKIP]
    f = random.choice(files)

    effective_bits = []
    with pod5.Reader(f) as r:
        for read in r.reads():
            sig = read.signal
            signal_range = int(sig.max()) - int(sig.min())
            if signal_range > 0:
                effective_bits.append(np.log2(signal_range))

    ax.hist(effective_bits, bins=50, color=color, alpha=0.8)
    ax.set_title(f"{name}\n({f.name})")
    ax.set_xlabel("Битовая ширина динамического диапазона рида (log₂(max − min))")
    ax.set_ylabel("Число ридов")
    ax.axvline(9,  color='black', linestyle='--', linewidth=1, label='9 бит')
    ax.axvline(10, color='gray',  linestyle='--', linewidth=1, label='10 бит')
    ax.axvline(13, color='red',   linestyle='--', linewidth=1, label='13 бит (АЦП)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("/content/effective_bits.png", dpi=150)
plt.show()
print("Сохранено: /content/effective_bits.png")

In [ ]:
N_READS = 250

def bit_width_stats(filepath, name, n_reads=N_READS):
    bits_list = []
    with pod5.Reader(filepath) as r:
        for i, read in enumerate(r.reads()):
            if i >= n_reads:
                break
            sig = read.signal
            rng = int(sig.max()) - int(sig.min())
            if rng > 0:
                bits_list.append(np.log2(rng))

    bits = np.array(bits_list)
    total = len(bits)

    print(f"\n{'='*40}")
    print(f"  {name}  ({filepath.name})")
    print(f"{'='*40}")
    print(f"Всего ридов: {total}")
    print(f"{'Бит':<6} {'Число ридов':<15} {'%'}")
    print("-" * 30)
    for b in range(7, 14):
        count = int(np.sum((bits >= b) & (bits < b+1)))
        print(f"{b}-{b+1:<4} {count:<15} {100*count/total:.1f}%")
    print(f"\nМедиана: {np.median(bits):.2f} бит")
    print(f"Среднее: {np.mean(bits):.2f} бит")
    print(f"<= 10 бит: {100*np.mean(bits <= 10):.1f}%")
    print(f"<=  9 бит: {100*np.mean(bits <= 9):.1f}%")

for name, d in DATASETS.items():
    files = [f for f in d.glob("*.pod5") if f.name not in SKIP]
    f = random.choice(files)
    bit_width_stats(f, name)

# Сжатие, анализ сжатия и подготовка к Basecoling

## --- Сжатие данных с потерями. Анализ результатов

### --- --- Глобальные переменные

In [43]:
METHOD_GROUPS = {
    "amplitude_quantization": [
        "lsb_k1", "lsb_k2", "lsb_k3",
        "quant_q2", "quant_q4", "quant_q8", "quant_q16",
        "bound_eps1", "bound_eps2", "bound_eps4", "bound_eps8",
    ],

    "predictive_coding": [
        "dpcm_q1", "dpcm_q2", "dpcm_q4", "dpcm_q8",
    ],

    "nonlinear_companding": [
        "mulaw_50", "mulaw_100", "mulaw_255",
    ],

    "run_length_after_quantization": [
        "rle_quant_q2", "rle_quant_q4", "rle_quant_q8", "rle_quant_q16",
    ],

    "temporal_resolution_reduction": [
        "paa_w2", "paa_w4", "paa_w8",
        "downsample_d2", "downsample_d4",
    ],

    "piecewise_approximation": [
        "deadband_eps1", "deadband_eps2", "deadband_eps4", "deadband_eps8", "deadband_eps16",
        "pla_eps1", "pla_eps2", "pla_eps4", "pla_eps8", "pla_eps16",
    ],

    "frequency_domain": [
        "fft_50", "fft_25", "fft_10",
    ],
}

In [44]:
METHOD_NAMES_ALL = ["original"] + [
    method
    for group_methods in METHOD_GROUPS.values()
    for method in group_methods
]

### --- --- Реализация

In [45]:
import numpy as np
import pod5
import zstandard as zstd
import pandas as pd

from pathlib import Path
from scipy import signal as sp


# ============================================================
# ОЦЕНКА CR_zstd И ОШИБКИ ВОССТАНОВЛЕНИЯ
# Единый реестр методов: method_id / group / family / param
# ============================================================

In [46]:
# ============================================================
# КОНФИГУРАЦИЯ
# ============================================================

DATASETS = {
    "Plasmid_R10_4_1": Path("/content/POD5/try2/downstream_subset_new/Plasmid_R10_4_1/original.pod5"),
    "E_coli_R9":       Path("/content/POD5/try2/downstream_subset_new/E_coli_R9/original.pod5"),
    "Bacterial_R10_4": Path("/content/POD5/try2/downstream_subset_new/Bacterial_R10_4/original.pod5"),
}

N_READS = 50
RNG_SEED = 42
ZSTD_LEVEL = 3

# Управление тяжёлыми/спорными методами
INCLUDE_DEADBAND = True
INCLUDE_PLA = True
INCLUDE_FFT = False

OUT_DIR = Path("/content/POD5/try2/compression_eval_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ZSTD_CTX = zstd.ZstdCompressor(level=ZSTD_LEVEL)

In [47]:
# ============================================================
# УТИЛИТЫ
# ============================================================

def compress_size(data: bytes) -> int:
    """Размер после zstd-сжатия."""
    return len(ZSTD_CTX.compress(data))


def signal_metrics(orig, dec):
    """
    Метрики ошибки восстановления сигнала.
    Значения измеряются в единицах исходного raw-сигнала.
    """
    d = orig.astype(np.float32) - dec.astype(np.float32)

    rmse = float(np.sqrt(np.mean(d ** 2)))
    mae = float(np.mean(np.abs(d)))
    maxerr = float(np.max(np.abs(d)))

    return rmse, mae, maxerr


def original_signal_bytes(sig):
    """Исходный сигнал считается как int16: 2 байта на отсчёт."""
    return sig.astype(np.int16).tobytes()

In [48]:
# ============================================================
# МЕТОДЫ-КОДЕКИ
# Каждый метод возвращает:
#   dec       — восстановленный int16-сигнал
#   enc_bytes — байтовое представление, которое затем сжимается zstd
# ============================================================

def lsb_zeroing_codec(sig, k):
    dec = (sig.astype(np.int32) >> k << k).astype(np.int16)
    enc_bytes = dec.tobytes()
    return dec, enc_bytes


def scalar_quant_codec(sig, q):
    dec = (
        np.round(sig.astype(np.float32) / q) * q
    ).clip(-32768, 32767).astype(np.int16)

    enc_bytes = dec.tobytes()
    return dec, enc_bytes


def bounded_quant_codec(sig, eps):
    """
    Равномерное квантование по интервалам шириной 2*eps.
    Значение заменяется серединой интервала.
    """
    q = 2 * eps

    dec = (
        np.floor(sig.astype(np.float32) / q) * q + eps
    ).clip(-32768, 32767).astype(np.int16)

    enc_bytes = dec.tobytes()
    return dec, enc_bytes


def paa_codec(sig, W):
    """
    Piecewise Aggregate Approximation.
    Храним одно среднее значение int16 на окно W.
    """
    sig = sig.astype(np.int16)
    n = (len(sig) // W) * W

    if len(sig) == 0:
        return sig.copy(), b""

    if n == 0:
        dec = sig.copy()
        return dec, dec.tobytes()

    enc = (
        np.round(sig[:n].astype(np.float32).reshape(-1, W).mean(axis=1))
    ).clip(-32768, 32767).astype(np.int16)

    dec = np.repeat(enc, W)

    if len(dec) < len(sig):
        dec = np.pad(dec, (0, len(sig) - len(dec)), mode="edge")

    dec = dec[:len(sig)].astype(np.int16)
    enc_bytes = enc.tobytes()

    return dec, enc_bytes


def downsample_codec(sig, D):
    """
    Прореживание сигнала с последующим восстановлением.
    Храним прореженный сигнал как int16.
    """
    sig = sig.astype(np.int16)

    if len(sig) == 0:
        return sig.copy(), b""

    s = sig.astype(np.float32)

    # На очень коротких ридах decimate может работать нестабильно.
    if len(s) < D * 4:
        dec = sig.copy()
        return dec, dec.tobytes()

    enc = sp.decimate(s, D, ftype="fir", zero_phase=True)
    enc_i16 = np.round(enc).clip(-32768, 32767).astype(np.int16)

    dec = sp.resample_poly(enc_i16.astype(np.float32), D, 1)

    if len(dec) < len(sig):
        dec = np.pad(dec, (0, len(sig) - len(dec)), mode="edge")

    dec = np.round(dec[:len(sig)]).clip(-32768, 32767).astype(np.int16)
    enc_bytes = enc_i16.tobytes()

    return dec, enc_bytes


def deadband_codec(sig, eps):
    """
    Deadband-кодирование.
    Сохраняются точки, где сигнал отклонился от последней сохранённой точки
    больше чем на eps. Восстановление выполняется линейной интерполяцией.

    Важно: это НЕ строгая bounded-error аппроксимация.
    Ошибка после интерполяции может быть больше eps.
    """
    sig = sig.astype(np.int16)

    if len(sig) == 0:
        return sig.copy(), b""

    s = sig.astype(np.float32)

    idx = [0]
    vals = [s[0]]
    last = s[0]

    for i in range(1, len(s)):
        if abs(s[i] - last) > eps:
            idx.append(i)
            vals.append(s[i])
            last = s[i]

    if idx[-1] != len(s) - 1:
        idx.append(len(s) - 1)
        vals.append(s[-1])

    idx_arr = np.array(idx, dtype=np.uint32)
    vals_arr = np.round(vals).clip(-32768, 32767).astype(np.int16)

    out = np.empty(len(s), dtype=np.float32)

    for k in range(len(idx_arr) - 1):
        i0 = int(idx_arr[k])
        i1 = int(idx_arr[k + 1])

        out[i0:i1] = np.linspace(
            vals_arr[k],
            vals_arr[k + 1],
            i1 - i0,
            endpoint=False
        )

    out[int(idx_arr[-1])] = vals_arr[-1]

    dec = np.round(out).clip(-32768, 32767).astype(np.int16)
    enc_bytes = idx_arr.tobytes() + vals_arr.tobytes()

    return dec, enc_bytes


def pla_codec(sig, eps):
    """
    Piecewise Linear Approximation.
    Сигнал разбивается на линейные сегменты.
    Храним индексы опорных точек и значения в этих точках.

    Метод может быть медленным на длинных ридах.
    """
    sig = sig.astype(np.int16)

    if len(sig) == 0:
        return sig.copy(), b""

    s = sig.astype(np.float32)
    n = len(s)

    idx = [0]
    vals = [s[0]]

    i = 0

    while i < n - 1:
        j = i + 1

        while j < n:
            t = np.arange(j - i + 1, dtype=np.float32) / (j - i)
            interp = s[i] + t * (s[j] - s[i])

            if np.max(np.abs(s[i:j + 1] - interp)) > eps:
                break

            j += 1

        end = max(j - 1, i + 1)

        idx.append(end)
        vals.append(s[end])

        i = end

    idx_arr = np.array(idx, dtype=np.uint32)
    vals_arr = np.round(vals).clip(-32768, 32767).astype(np.int16)

    out = np.empty(n, dtype=np.float32)

    for k in range(len(idx_arr) - 1):
        i0 = int(idx_arr[k])
        i1 = int(idx_arr[k + 1])

        out[i0:i1] = np.linspace(
            vals_arr[k],
            vals_arr[k + 1],
            i1 - i0,
            endpoint=False
        )

    out[int(idx_arr[-1])] = vals_arr[-1]

    dec = np.round(out).clip(-32768, 32767).astype(np.int16)
    enc_bytes = idx_arr.tobytes() + vals_arr.tobytes()

    return dec, enc_bytes


def mulaw_codec(sig, mu=255, q=256):
    """
    μ-law-компандирование.
    При q=256 код хранится как uint8.
    """
    if q > 256:
        raise ValueError("q должно быть не больше 256, так как используется uint8")

    sig = sig.astype(np.int16)

    if len(sig) == 0:
        return sig.copy(), b""

    s = sig.astype(np.float32)

    offset = np.float32(np.median(s))
    centered = s - offset

    xmax = np.float32(max(float(np.max(np.abs(centered))), 1.0))

    y = (
        np.sign(centered)
        * np.log1p(mu * np.abs(centered) / xmax)
        / np.log1p(mu)
    )

    levels = q - 1

    enc = np.clip(
        np.rint((y + 1.0) * levels / 2.0),
        0,
        levels
    ).astype(np.uint8)

    yq = enc.astype(np.float32) * 2.0 / levels - 1.0

    xhat = (
        np.sign(yq)
        * xmax
        * (np.expm1(np.abs(yq) * np.log1p(mu)) / mu)
        + offset
    )

    dec = np.round(xhat).clip(-32768, 32767).astype(np.int16)

    overhead = np.array([offset, xmax], dtype=np.float32)
    enc_bytes = overhead.tobytes() + enc.tobytes()

    return dec, enc_bytes


def dpcm_closed_loop_codec(sig, q):
    """
    DPCM с замкнутым контуром.
    Храним первое значение сигнала и индексы квантованных остатков.

    Остаток восстанавливается как:
        residual_hat = residual_index * q
    """
    sig = sig.astype(np.int16)

    if len(sig) == 0:
        return sig.copy(), b""

    s = sig.astype(np.int32)
    n = len(s)

    recon = np.empty(n, dtype=np.int32)
    recon[0] = s[0]

    q_idx = np.empty(n - 1, dtype=np.int32)

    for i in range(1, n):
        pred = recon[i - 1]
        residual = s[i] - pred

        idx = int(np.round(residual / q))
        q_idx[i - 1] = idx

        recon[i] = np.clip(pred + idx * q, -32768, 32767)

    dec = recon.astype(np.int16)

    first = np.array([s[0]], dtype=np.int16)

    if q_idx.size == 0:
        q_idx_payload = np.array([], dtype=np.int16)
        dtype_marker = np.array([16], dtype=np.uint8)
    elif (
        q_idx.min() >= np.iinfo(np.int16).min
        and q_idx.max() <= np.iinfo(np.int16).max
    ):
        q_idx_payload = q_idx.astype(np.int16)
        dtype_marker = np.array([16], dtype=np.uint8)
    else:
        q_idx_payload = q_idx.astype(np.int32)
        dtype_marker = np.array([32], dtype=np.uint8)

    enc_bytes = first.tobytes() + dtype_marker.tobytes() + q_idx_payload.tobytes()

    return dec, enc_bytes


def rle_quant_codec(sig, q):
    """
    Квантование + RLE.
    Храним пары:
        value: int16
        run_length: uint16
    """
    sig = sig.astype(np.int16)

    if len(sig) == 0:
        return sig.copy(), b""

    q_sig = (
        np.round(sig.astype(np.float32) / q) * q
    ).clip(-32768, 32767).astype(np.int16)

    vals = []
    lens = []

    cur_val = q_sig[0]
    cur_len = 1

    for v in q_sig[1:]:
        if v == cur_val and cur_len < 65535:
            cur_len += 1
        else:
            vals.append(cur_val)
            lens.append(cur_len)

            cur_val = v
            cur_len = 1

    vals.append(cur_val)
    lens.append(cur_len)

    vals_arr = np.array(vals, dtype=np.int16)
    lens_arr = np.array(lens, dtype=np.uint16)

    dec = np.repeat(vals_arr, lens_arr.astype(np.int32)).astype(np.int16)
    dec = dec[:len(sig)]

    enc_bytes = vals_arr.tobytes() + lens_arr.tobytes()

    return dec, enc_bytes


def fft_codec(sig, keep_ratio):
    """
    Экспериментальный FFT-вариант.
    Восстановление выполняется из тех же float32-коэффициентов,
    которые считаются сохранёнными.
    """
    sig = sig.astype(np.int16)

    if len(sig) == 0:
        return sig.copy(), b""

    s = sig.astype(np.float32)
    coeffs = np.fft.rfft(s)

    n_keep = max(1, int(len(coeffs) * keep_ratio))
    top_idx = np.argsort(np.abs(coeffs))[-n_keep:]

    idx_arr = top_idx.astype(np.uint32)
    re_arr = coeffs[top_idx].real.astype(np.float32)
    im_arr = coeffs[top_idx].imag.astype(np.float32)

    sparse = np.zeros_like(coeffs, dtype=np.complex64)
    sparse[idx_arr] = re_arr.astype(np.float32) + 1j * im_arr.astype(np.float32)

    dec = np.round(
        np.fft.irfft(sparse, n=len(s))
    ).clip(-32768, 32767).astype(np.int16)

    enc_bytes = idx_arr.tobytes() + re_arr.tobytes() + im_arr.tobytes()

    return dec, enc_bytes

In [49]:
# ============================================================
# РЕЕСТР МЕТОДОВ
# ============================================================

def build_method_specs():
    """
    Единый реестр методов для скрининга сжатия.

    Поля:
      method_id — уникальное имя метода, совпадает с именем POD5/FASTQ
      group     — группа по принципу работы
      family    — семейство метода
      param     — параметры
      fn        — функция, возвращающая (dec, enc_bytes)
    """
    methods = []

    # --------------------------------------------------------
    # 1. Равномерное квантование амплитуды
    # --------------------------------------------------------

    for k in [1, 2, 3]:
        methods.append({
            "method_id": f"lsb_k{k}",
            "group": "amplitude_quantization",
            "family": "LSB zeroing",
            "param": f"k={k}",
            "fn": lambda s, k=k: lsb_zeroing_codec(s, k),
        })

    for q in [2, 4, 8, 16]:
        methods.append({
            "method_id": f"quant_q{q}",
            "group": "amplitude_quantization",
            "family": "Uniform quantization",
            "param": f"q={q}",
            "fn": lambda s, q=q: scalar_quant_codec(s, q),
        })

    for eps in [1, 2, 4, 8]:
        methods.append({
            "method_id": f"bound_eps{eps}",
            "group": "amplitude_quantization",
            "family": "Bounded quantization",
            "param": f"eps={eps}",
            "fn": lambda s, eps=eps: bounded_quant_codec(s, eps),
        })

    # --------------------------------------------------------
    # 2. Предиктивное кодирование
    # --------------------------------------------------------

    for q in [1, 2, 4, 8]:
        methods.append({
            "method_id": f"dpcm_q{q}",
            "group": "predictive_coding",
            "family": "DPCM",
            "param": f"q={q}",
            "fn": lambda s, q=q: dpcm_closed_loop_codec(s, q),
        })

    # --------------------------------------------------------
    # 3. Нелинейное квантование / компандирование
    # --------------------------------------------------------

    for mu in [50, 100, 255]:
        methods.append({
            "method_id": f"mulaw_{mu}",
            "group": "nonlinear_companding",
            "family": "Mu-law",
            "param": f"mu={mu}, q=256",
            "fn": lambda s, mu=mu: mulaw_codec(s, mu=mu, q=256),
        })

    # --------------------------------------------------------
    # 4. Квантование + RLE
    # --------------------------------------------------------

    for q in [2, 4, 8, 16]:
        methods.append({
            "method_id": f"rle_quant_q{q}",
            "group": "run_length_after_quantization",
            "family": "RLE + quantization",
            "param": f"q={q}",
            "fn": lambda s, q=q: rle_quant_codec(s, q),
        })

    # --------------------------------------------------------
    # 5. Снижение временного разрешения
    # --------------------------------------------------------

    for W in [2, 4, 8]:
        methods.append({
            "method_id": f"paa_w{W}",
            "group": "temporal_resolution_reduction",
            "family": "PAA",
            "param": f"W={W}",
            "fn": lambda s, W=W: paa_codec(s, W),
        })

    for D in [2, 4]:
        methods.append({
            "method_id": f"downsample_d{D}",
            "group": "temporal_resolution_reduction",
            "family": "Downsample",
            "param": f"D={D}",
            "fn": lambda s, D=D: downsample_codec(s, D),
        })

    # --------------------------------------------------------
    # 6. Кусочная аппроксимация
    # --------------------------------------------------------

    if INCLUDE_DEADBAND:
        for eps in [1, 2, 4, 8, 16]:
            methods.append({
                "method_id": f"deadband_eps{eps}",
                "group": "piecewise_approximation",
                "family": "Deadband",
                "param": f"eps={eps}",
                "fn": lambda s, eps=eps: deadband_codec(s, eps),
            })

    if INCLUDE_PLA:
        for eps in [1, 2, 4, 8, 16]:
            methods.append({
                "method_id": f"pla_eps{eps}",
                "group": "piecewise_approximation",
                "family": "PLA",
                "param": f"eps={eps}",
                "fn": lambda s, eps=eps: pla_codec(s, eps),
            })

    # --------------------------------------------------------
    # 7. Частотная аппроксимация
    # --------------------------------------------------------

    if INCLUDE_FFT:
        for keep_ratio in [0.5, 0.25, 0.1]:
            percent = int(keep_ratio * 100)
            methods.append({
                "method_id": f"fft_{percent}",
                "group": "frequency_domain",
                "family": "FFT",
                "param": f"keep={percent}%",
                "fn": lambda s, keep_ratio=keep_ratio: fft_codec(s, keep_ratio),
            })

    # --------------------------------------------------------
    # Убираем слишком медленные методы из общего реестра
    # --------------------------------------------------------

    methods = [
        m for m in methods
        if not m["method_id"].startswith("deadband_")
    ]

    return methods


METHOD_SPECS = build_method_specs()

METHOD_NAMES_ALL = ["original"] + [
    m["method_id"] for m in METHOD_SPECS
]

METHOD_GROUPS = {}

for m in METHOD_SPECS:
    METHOD_GROUPS.setdefault(m["group"], []).append(m["method_id"])

print(f"Методов для скрининга: {len(METHOD_SPECS)}")
print("METHOD_NAMES_ALL:")
print(METHOD_NAMES_ALL)

print("\nMETHOD_GROUPS:")
for group, names in METHOD_GROUPS.items():
    print(f"  {group}: {len(names)} методов")

Методов для скрининга: 32
METHOD_NAMES_ALL:
['original', 'lsb_k1', 'lsb_k2', 'lsb_k3', 'quant_q2', 'quant_q4', 'quant_q8', 'quant_q16', 'bound_eps1', 'bound_eps2', 'bound_eps4', 'bound_eps8', 'dpcm_q1', 'dpcm_q2', 'dpcm_q4', 'dpcm_q8', 'mulaw_50', 'mulaw_100', 'mulaw_255', 'rle_quant_q2', 'rle_quant_q4', 'rle_quant_q8', 'rle_quant_q16', 'paa_w2', 'paa_w4', 'paa_w8', 'downsample_d2', 'downsample_d4', 'pla_eps1', 'pla_eps2', 'pla_eps4', 'pla_eps8', 'pla_eps16']

METHOD_GROUPS:
  amplitude_quantization: 11 методов
  predictive_coding: 4 методов
  nonlinear_companding: 3 методов
  run_length_after_quantization: 4 методов
  temporal_resolution_reduction: 5 методов
  piecewise_approximation: 5 методов


In [50]:
# ============================================================
# ЭКСПЕРИМЕНТ
# ============================================================

rng = np.random.default_rng(RNG_SEED)

all_results = []

for ds_name, ds_path in DATASETS.items():
    print(f"\n{'=' * 80}")
    print(f"DATASET: {ds_name}")
    print(f"{'=' * 80}")

    with pod5.Reader(ds_path) as reader:
        sigs = [read.signal.astype(np.int16) for read in reader.reads()]

    if len(sigs) == 0:
        print("  Нет ридов, пропуск.")
        continue

    chosen = rng.choice(
        len(sigs),
        size=min(N_READS, len(sigs)),
        replace=False
    )

    signals = [sigs[i] for i in chosen]

    read_lengths = np.array([len(s) for s in signals])

    print(f"  Ридов использовано: {len(signals)}")
    print(f"  Медианная длина рида: {int(np.median(read_lengths))}")
    print(f"  Суммарное число отсчётов: {int(read_lengths.sum())}")

    orig_raw_bytes_total = sum(len(s) * 2 for s in signals)
    orig_zstd_bytes_total = sum(compress_size(original_signal_bytes(s)) for s in signals)

    orig_zstd_cr = orig_raw_bytes_total / orig_zstd_bytes_total

    print(
        f"  Оригинал: "
        f"{orig_raw_bytes_total / 1e6:.3f} MB raw → "
        f"{orig_zstd_bytes_total / 1e6:.3f} MB zstd{ZSTD_LEVEL} "
        f"(CR={orig_zstd_cr:.2f}x)"
    )

    dataset_results = []

    for method in METHOD_SPECS:
        method_id = method["method_id"]
        group = method["group"]
        family = method["family"]
        param = method["param"]
        fn = method["fn"]

        encoded_raw_total = 0
        encoded_zstd_total = 0

        rmses = []
        maes = []
        maxerrs = []

        for sig in signals:
            dec, enc_bytes = fn(sig)

            rmse, mae, maxerr = signal_metrics(sig, dec)

            encoded_raw_total += len(enc_bytes)
            encoded_zstd_total += compress_size(enc_bytes)

            rmses.append(rmse)
            maes.append(mae)
            maxerrs.append(maxerr)

        result = {
            "dataset": ds_name,

            "method_id": method_id,
            "group": group,
            "family": family,
            "param": param,

            "orig_raw_bytes": orig_raw_bytes_total,
            "orig_zstd_bytes": orig_zstd_bytes_total,

            "encoded_raw_bytes": encoded_raw_total,
            "encoded_zstd_bytes": encoded_zstd_total,

            # Коэффициент сжатия относительно исходного int16-сигнала
            "cr_zstd_vs_raw": orig_raw_bytes_total / encoded_zstd_total,

            # Выигрыш относительно простого zstd на исходном сигнале
            # > 1.0 означает, что метод лучше baseline zstd.
            "gain_vs_original_zstd": orig_zstd_bytes_total / encoded_zstd_total,

            "rmse_mean": float(np.mean(rmses)),
            "mae_mean": float(np.mean(maes)),
            "maxerr_mean": float(np.mean(maxerrs)),
            "maxerr_global": float(np.max(maxerrs)),
        }

        dataset_results.append(result)
        all_results.append(result)

    df_ds = pd.DataFrame(dataset_results)

    df_print = df_ds.sort_values(
        by=["gain_vs_original_zstd", "cr_zstd_vs_raw"],
        ascending=False
    )

    print("\n  Результаты, отсортировано по выигрышу относительно zstd на исходном сигнале:")
    print(
        df_print[
            [
                "group",
                "family",
                "method_id",
                "param",
                "cr_zstd_vs_raw",
                "gain_vs_original_zstd",
                "rmse_mean",
                "mae_mean",
                "maxerr_mean",
                "maxerr_global",
            ]
        ].to_string(
            index=False,
            formatters={
                "cr_zstd_vs_raw": "{:.2f}".format,
                "gain_vs_original_zstd": "{:.2f}".format,
                "rmse_mean": "{:.2f}".format,
                "mae_mean": "{:.2f}".format,
                "maxerr_mean": "{:.1f}".format,
                "maxerr_global": "{:.1f}".format,
            }
        )
    )


DATASET: Plasmid_R10_4_1


FileNotFoundError: Failed to open pod5 file at: /content/POD5/try2/downstream_subset_new/Plasmid_R10_4_1/original.pod5

### --- --- Сохранение данных

In [ ]:
# ============================================================
# СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ============================================================

df_all = pd.DataFrame(all_results)

out_csv = OUT_DIR / "lossy_signal_compression_eval_zstd.csv"
df_all.to_csv(out_csv, index=False)

# Дополнительно сохраняем сводку по группам:
# для каждой группы и датасета берём лучший метод по gain_vs_original_zstd.
df_best_by_group = (
    df_all
    .sort_values(["dataset", "group", "gain_vs_original_zstd"], ascending=[True, True, False])
    .groupby(["dataset", "group"], as_index=False)
    .head(1)
)

out_best_csv = OUT_DIR / "best_methods_by_group_zstd.csv"
df_best_by_group.to_csv(out_best_csv, index=False)

# Сохраняем реестр методов
df_methods = pd.DataFrame([
    {
        "method_id": m["method_id"],
        "group": m["group"],
        "family": m["family"],
        "param": m["param"],
    }
    for m in METHOD_SPECS
])

out_methods_csv = OUT_DIR / "method_registry.csv"
df_methods.to_csv(out_methods_csv, index=False)

print("\nГотово.")
print(f"Полные результаты сохранены: {out_csv}")
print(f"Лучшие методы по группам сохранены: {out_best_csv}")
print(f"Реестр методов сохранён: {out_methods_csv}")

## --- Подготовка данных для basecolling

In [ ]:
from pathlib import Path
import shutil
import random
import numpy as np

# ============================================================
# ПОДГОТОВКА ПОДНАБОРА ДЛЯ DOWNSTREAM-ЭКСПЕРИМЕНТА
# POD5 → аппроксимированные POD5 → basecalling
# ============================================================

BASE_DIR = Path("/content/POD5")

DOWNSTREAM_SUBSET_DIR = BASE_DIR / "try2" / "downstream_subset_new"
DOWNSTREAM_SUBSET_DIR.mkdir(parents=True, exist_ok=True)

# Если нужно пересоздать всё с нуля, поставить True.
OVERWRITE_DOWNSTREAM_SUBSET = False

if OVERWRITE_DOWNSTREAM_SUBSET and DOWNSTREAM_SUBSET_DIR.exists():
    shutil.rmtree(DOWNSTREAM_SUBSET_DIR)
    DOWNSTREAM_SUBSET_DIR.mkdir(parents=True, exist_ok=True)

# Битые или проблемные POD5-файлы, которые исключаются из выборки
SKIP_FILES = {
    "barcode02_r0barcode02b48_0.pod5",
}

SOURCE_DATASETS = [
    ("Bacterial_R10_4",   BASE_DIR / "Bacterial R10.4 data"),
    ("Plasmid_R10_4_1",  BASE_DIR / "plasmid_2025.04"),
    ("E_coli_R9",        BASE_DIR / "R9 data of E. coli (ERR9127551)"),
]

# Число ридов, которые берём для downstream-проверки
N_READS_DOWNSTREAM = 2000
# Если True — пересоздаёт уже существующие method.pod5
OVERWRITE_POD5 = True

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

In [ ]:
# ============================================================
# МЕТОДЫ ДЛЯ СОЗДАНИЯ POD5-ФАЙЛОВ
# Использует тот же реестр METHOD_SPECS, что и в оценке CR_zstd
# ============================================================

def original_transform(sig):
    return sig.astype(np.int16).copy()


def transform_from_codec(codec_fn):
    """
    Преобразует codec-функцию вида:
        codec_fn(sig) -> (dec, enc_bytes)

    в функцию для записи POD5:
        transform(sig) -> dec
    """
    def transform(sig):
        dec, _ = codec_fn(sig.astype(np.int16))
        return np.clip(dec, -32768, 32767).astype(np.int16)

    return transform


METHODS_FOR_POD5 = {
    "original": original_transform
}

for spec in METHOD_SPECS:
    method_id = spec["method_id"]
    codec_fn = spec["fn"]

    METHODS_FOR_POD5[method_id] = transform_from_codec(codec_fn)


METHOD_NAMES_FOR_POD5 = list(METHODS_FOR_POD5.keys())

print(f"Методов для создания POD5: {len(METHOD_NAMES_FOR_POD5)}")
print(METHOD_NAMES_FOR_POD5)

In [ ]:
# ============================================================
# ОБЁРТКА ДЛЯ РИДА
# ============================================================

class ReadData:
    """
    Копия нужных полей POD5-рида.
    Нужна, чтобы один раз выбрать риды, сохранить их в памяти
    и затем записать одинаковый набор read_id для всех методов.
    """

    def __init__(self, read):
        self.read_id = read.read_id
        self.end_reason = read.end_reason
        self.calibration = read.calibration
        self.pore = read.pore
        self.read_number = read.read_number
        self.start_sample = read.start_sample
        self.median_before = read.median_before
        self.num_minknow_events = read.num_minknow_events
        self.run_info = read.run_info

        # Эти поля могут быть deprecated в pod5, поэтому читаем осторожно.
        self.tracked_scaling = getattr(read, "tracked_scaling", None)
        self.predicted_scaling = getattr(read, "predicted_scaling", None)
        self.num_reads_since_mux_change = getattr(read, "num_reads_since_mux_change", None)
        self.time_since_mux_change = getattr(read, "time_since_mux_change", None)

        self.signal = read.signal.astype(np.int16).copy()


In [ ]:
# ============================================================
# МЕТОДЫ ДЛЯ ЗАПИСИ POD5
# Используем METHOD_SPECS из блока оценки CR_zstd
# ============================================================

def original_transform(sig):
    return sig.astype(np.int16).copy()


def transform_from_codec(codec_fn):
    """
    METHOD_SPECS содержит codec-функции вида:
        codec_fn(sig) -> (dec, enc_bytes)

    Для записи POD5 нужен только восстановленный сигнал dec.
    """
    def transform(sig):
        dec, _ = codec_fn(sig.astype(np.int16))
        return np.clip(dec, -32768, 32767).astype(np.int16)

    return transform


# ============================================================
# ИСКЛЮЧАЕМ СЛИШКОМ МЕДЛЕННЫЕ МЕТОДЫ
# ============================================================

EXCLUDED_PREFIXES_FOR_POD5 = (
    "deadband_",
    "pla_",
)

METHODS_FOR_POD5 = {
    "original": original_transform
}

for spec in METHOD_SPECS:
    method_id = spec["method_id"]

    if method_id.startswith(EXCLUDED_PREFIXES_FOR_POD5):
        continue

    METHODS_FOR_POD5[method_id] = transform_from_codec(spec["fn"])


METHOD_NAMES_FOR_POD5 = list(METHODS_FOR_POD5.keys())

print(f"Методов для создания POD5: {len(METHOD_NAMES_FOR_POD5)}")
print(METHOD_NAMES_FOR_POD5)

print()
print("Проверка, что deadband/pla убраны:")
print([
    m for m in METHOD_NAMES_FOR_POD5
    if m.startswith("deadband_") or m.startswith("pla_")
])

In [ ]:
# ============================================================
# ЗАПИСЬ POD5
# ============================================================

for dataset_name, pod5_dir in SOURCE_DATASETS:
    print(f"\n{'=' * 80}")
    print(f"DATASET: {dataset_name}")
    print(f"{'=' * 80}")

    files = sorted([
        f for f in pod5_dir.glob("*.pod5")
        if f.name not in SKIP_FILES
    ])

    print(f"  POD5-файлов найдено: {len(files)}")

    if not files:
        print("  Нет файлов, пропуск.")
        continue

    # --------------------------------------------------------
    # 1. Собираем все доступные read_id
    # --------------------------------------------------------

    all_entries = []

    for f in files:
        with pod5.Reader(f) as reader:
            for read in reader.reads():
                # Храним пару file + read_id, чтобы избежать проблем
                # при гипотетическом совпадении read_id в разных файлах.
                all_entries.append((str(f), str(read.read_id)))

    if not all_entries:
        print("  Нет ридов, пропуск.")
        continue

    n_select = min(N_READS_DOWNSTREAM, len(all_entries))

    selected_entries = set(
        random.sample(all_entries, n_select)
    )

    selected_by_file = {}

    for file_path, read_id in selected_entries:
        selected_by_file.setdefault(file_path, set()).add(read_id)

    print(f"  Доступно ридов: {len(all_entries)}")
    print(f"  Выбрано ридов: {n_select}")

    # --------------------------------------------------------
    # 2. Читаем выбранные риды в память
    # --------------------------------------------------------

    selected_reads = []

    for f in files:
        f_key = str(f)

        if f_key not in selected_by_file:
            continue

        needed_ids = selected_by_file[f_key]

        with pod5.Reader(f) as reader:
            for read in reader.reads():
                if str(read.read_id) in needed_ids:
                    selected_reads.append(ReadData(read))

        if len(selected_reads) >= n_select:
            break

    print(f"  Прочитано выбранных ридов: {len(selected_reads)}")

    if len(selected_reads) == 0:
        print("  Выбранные риды не прочитаны, пропуск.")
        continue

    # --------------------------------------------------------
    # 3. Создаём POD5 для каждого метода
    # --------------------------------------------------------

    dataset_dir = DOWNSTREAM_SUBSET_DIR / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)

    for method_name, transform in METHODS_FOR_POD5.items():
        out_path = dataset_dir / f"{method_name}.pod5"

        if out_path.exists() and not OVERWRITE_POD5:
            print(f"  {method_name:<24} уже есть, пропускаем")
            continue

        if out_path.exists() and OVERWRITE_POD5:
            out_path.unlink()

        print(f"  {method_name:<24}", end=" ", flush=True)

        count = 0

        with pod5.Writer(out_path) as writer:
            for rd in selected_reads:
                transformed = transform(rd.signal)
                transformed = np.clip(transformed, -32768, 32767).astype(np.int16)

                read_kwargs = dict(
                    read_id=rd.read_id,
                    end_reason=rd.end_reason,
                    calibration=rd.calibration,
                    pore=rd.pore,
                    read_number=rd.read_number,
                    start_sample=rd.start_sample,
                    median_before=rd.median_before,
                    num_minknow_events=rd.num_minknow_events,
                    run_info=rd.run_info,
                    signal=transformed,
                )

                # Эти поля добавляем только если они реально есть.
                # Если pod5.Read в твоей версии их не принимает, можно убрать этот блок.
                if rd.tracked_scaling is not None:
                    read_kwargs["tracked_scaling"] = rd.tracked_scaling
                if rd.predicted_scaling is not None:
                    read_kwargs["predicted_scaling"] = rd.predicted_scaling
                if rd.num_reads_since_mux_change is not None:
                    read_kwargs["num_reads_since_mux_change"] = rd.num_reads_since_mux_change
                if rd.time_since_mux_change is not None:
                    read_kwargs["time_since_mux_change"] = rd.time_since_mux_change

                writer.add_read(pod5.Read(**read_kwargs))
                count += 1

        print(f"{count} ридов")

print("\nВсе downstream POD5-файлы созданы.")

In [ ]:
from pathlib import Path
import shutil
import os

# ============================================================
# КОПИРОВАНИЕ DOWNSTREAM POD5 НА GOOGLE DRIVE
# ============================================================

# Если Google Drive ещё не подключён
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive уже может быть подключён или Colab недоступен:")
    print(e)

# Откуда копируем
try:
    SRC_DIR = DOWNSTREAM_SUBSET_DIR
except NameError:
    SRC_DIR = Path("/content/POD5/try2/downstream_subset_new")

SRC_DIR = Path(SRC_DIR)

# Куда копируем на Google Drive
DST_DIR = Path("/content/drive/MyDrive/POD5/try2/downstream_subset_new")

# Если True — перезаписывать файлы на диске.
# Если False — уже существующие файлы того же размера будут пропускаться.
OVERWRITE_ON_DRIVE = True

if not SRC_DIR.exists():
    raise FileNotFoundError(f"Исходная папка не найдена: {SRC_DIR}")

DST_DIR.mkdir(parents=True, exist_ok=True)

print("Источник:")
print(SRC_DIR)
print()
print("Папка на Google Drive:")
print(DST_DIR)
print()

copied = 0
skipped = 0
overwritten = 0
size_copied = 0

pod5_files = sorted(SRC_DIR.rglob("*.pod5"))

print(f"POD5-файлов найдено: {len(pod5_files)}")
print()

for src_path in pod5_files:
    rel_path = src_path.relative_to(SRC_DIR)
    dst_path = DST_DIR / rel_path

    dst_path.parent.mkdir(parents=True, exist_ok=True)

    src_size = src_path.stat().st_size

    if dst_path.exists():
        dst_size = dst_path.stat().st_size

        if not OVERWRITE_ON_DRIVE and dst_size == src_size:
            print(f"SKIP   {rel_path} ({src_size / 1e6:.1f} MB)")
            skipped += 1
            continue

        if OVERWRITE_ON_DRIVE:
            dst_path.unlink()
            overwritten += 1
        else:
            print(
                f"SKIP   {rel_path} "
                f"(файл уже есть, но размер отличается: "
                f"{dst_size / 1e6:.1f} MB vs {src_size / 1e6:.1f} MB)"
            )
            skipped += 1
            continue

    shutil.copy2(src_path, dst_path)

    print(f"COPY   {rel_path} ({src_size / 1e6:.1f} MB)")
    copied += 1
    size_copied += src_size

print()
print("Копирование завершено.")
print(f"Скопировано файлов: {copied}")
print(f"Пропущено файлов:   {skipped}")
print(f"Перезаписано:       {overwritten}")
print(f"Скопировано данных: {size_copied / 1e9:.2f} GB")
print()
print("Папка на Google Drive:")
print(DST_DIR)

# Basecolling

### --- Оценка коэффициента сжатия и ошибки восстановления для методов аппроксимации сырого сигнала

In [ ]:
import numpy as np
import pod5
from pathlib import Path

DOWNSTREAM_SUBSET_DIR = Path("/content/POD5/try2/downstream_subset_new")

DATASET_ORIGINALS = {
    "Plasmid_R10_4_1": DOWNSTREAM_SUBSET_DIR / "Plasmid_R10_4_1" / "original.pod5",
    "E_coli_R9":       DOWNSTREAM_SUBSET_DIR / "E_coli_R9" / "original.pod5",
    "Bacterial_R10_4": DOWNSTREAM_SUBSET_DIR / "Bacterial_R10_4" / "original.pod5",
}

for dataset, path in DATASET_ORIGINALS.items():
    if not path.exists():
        print(f"\n{dataset}: файл не найден: {path}")
        continue

    with pod5.Reader(path) as reader:
        lengths = np.array([len(read.signal) for read in reader.reads()], dtype=np.int64)

    if len(lengths) == 0:
        print(f"\n{dataset}: нет ридов")
        continue

    print(f"\n{dataset}: {len(lengths)} ридов")
    print(f"  Мин: {lengths.min()}, Макс: {lengths.max()}")
    print(f"  Медиана: {np.median(lengths):.0f}, Среднее: {lengths.mean():.0f}")
    print(f"  < 5000 отсчётов: {(lengths < 5000).sum()} ({(lengths < 5000).mean()*100:.1f}%)")
    print(f"  < 10000 отсчётов: {(lengths < 10000).sum()} ({(lengths < 10000).mean()*100:.1f}%)")
    print(f"  >= 10000 отсчётов: {(lengths >= 10000).sum()} ({(lengths >= 10000).mean()*100:.1f}%)")

In [ ]:
import pod5
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# ============================================================
# АНАЛИЗ ЭНТРОПИИ СИГНАЛА И ДЕЛЬТ В СОЗДАННЫХ POD5
# ============================================================

SUBSET_DIR = Path("/content/POD5/try2/downstream_subset_new")
OUT_DIR = Path("/content/POD5/try2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS_TO_ANALYZE = [
    "Plasmid_R10_4_1",
    "E_coli_R9",
    "Bacterial_R10_4",
]

# Берём тот же список методов, который использовался при создании POD5.
# Важно: METHOD_NAMES_FOR_POD5 должен быть определён выше в ноутбуке.
try:
    METHODS_TO_ANALYZE = list(METHOD_NAMES_FOR_POD5)
except NameError:
    raise NameError(
        "Перед этим блоком должен быть определён METHOD_NAMES_FOR_POD5. "
        "Проверь, что выше выполнен блок с METHOD_SPECS / METHOD_NAMES_FOR_POD5."
    )

# На случай если original по ошибке не попал в общий список
if "original" not in METHODS_TO_ANALYZE:
    METHODS_TO_ANALYZE = ["original"] + METHODS_TO_ANALYZE


# ============================================================
# ФУНКЦИИ
# ============================================================

def entropy_from_count_array(counts) -> float:
    """
    Энтропия по массиву частот.

    H = -sum(p_i * log2(p_i))
    """
    counts = np.asarray(counts, dtype=np.float64)
    total = counts.sum()

    if total <= 0:
        return np.nan

    p = counts / total
    p = p[p > 0]

    return float(-(p * np.log2(p)).sum())


def entropy_from_counter(counter: Counter) -> float:
    """
    Энтропия по Counter(value -> count).
    """
    if not counter:
        return np.nan

    return entropy_from_count_array(list(counter.values()))


def update_counter_from_unique(counter: Counter, values, counts) -> None:
    """
    Добавляет в Counter частоты, полученные через np.unique(..., return_counts=True).
    """
    for v, c in zip(values, counts):
        counter[int(v)] += int(c)


def collect_entropies_from_pod5(path: Path) -> dict | None:
    """
    Считает энтропийные характеристики для одного POD5-файла.

    Возвращает:
        n_reads
        total_signal_samples
        total_delta_samples

        mean_signal_entropy
        weighted_signal_entropy
        global_signal_entropy

        mean_delta_entropy
        weighted_delta_entropy
        global_delta_entropy

    Пояснение:
        mean_*     — обычное среднее по ридам;
        weighted_* — среднее по ридам с весом по длине рида;
        global_*   — энтропия по всем отсчётам/дельтам вместе.
    """
    per_read_signal_entropy = []
    per_read_delta_entropy = []

    signal_lengths = []
    delta_lengths = []

    global_signal_counts = Counter()
    global_delta_counts = Counter()

    total_signal_samples = 0
    total_delta_samples = 0

    with pod5.Reader(path) as reader:
        for read in reader.reads():
            sig = np.asarray(read.signal, dtype=np.int16)

            if len(sig) == 0:
                continue

            # ------------------------------------------------
            # Энтропия исходных значений сигнала
            # ------------------------------------------------
            values, counts = np.unique(sig, return_counts=True)

            h_sig = entropy_from_count_array(counts)

            per_read_signal_entropy.append(h_sig)
            signal_lengths.append(len(sig))
            total_signal_samples += len(sig)

            update_counter_from_unique(
                global_signal_counts,
                values,
                counts
            )

            # ------------------------------------------------
            # Энтропия дельт
            # ------------------------------------------------
            if len(sig) >= 2:
                delta = np.diff(sig.astype(np.int32))

                values_d, counts_d = np.unique(
                    delta,
                    return_counts=True
                )

                h_delta = entropy_from_count_array(counts_d)

                per_read_delta_entropy.append(h_delta)
                delta_lengths.append(len(delta))
                total_delta_samples += len(delta)

                update_counter_from_unique(
                    global_delta_counts,
                    values_d,
                    counts_d
                )

    if len(per_read_signal_entropy) == 0:
        return None

    per_read_signal_entropy = np.asarray(
        per_read_signal_entropy,
        dtype=np.float64
    )

    signal_lengths = np.asarray(
        signal_lengths,
        dtype=np.float64
    )

    per_read_delta_entropy = np.asarray(
        per_read_delta_entropy,
        dtype=np.float64
    )

    delta_lengths = np.asarray(
        delta_lengths,
        dtype=np.float64
    )

    result = {
        "n_reads": int(len(per_read_signal_entropy)),
        "total_signal_samples": int(total_signal_samples),
        "total_delta_samples": int(total_delta_samples),

        "mean_signal_entropy": float(np.mean(per_read_signal_entropy)),
        "weighted_signal_entropy": float(
            np.average(
                per_read_signal_entropy,
                weights=signal_lengths
            )
        ),
        "global_signal_entropy": entropy_from_counter(
            global_signal_counts
        ),

        "mean_delta_entropy": (
            float(np.mean(per_read_delta_entropy))
            if len(per_read_delta_entropy) > 0
            else np.nan
        ),
        "weighted_delta_entropy": (
            float(
                np.average(
                    per_read_delta_entropy,
                    weights=delta_lengths
                )
            )
            if len(per_read_delta_entropy) > 0
            else np.nan
        ),
        "global_delta_entropy": entropy_from_counter(
            global_delta_counts
        ),
    }

    return result


def add_entropy_deltas_vs_original(dataset_rows: list[dict]) -> list[dict]:
    """
    Добавляет разности энтропий относительно метода original внутри одного датасета.
    """
    original_row = None

    for row in dataset_rows:
        if row["method_id"] == "original":
            original_row = row
            break

    delta_metrics = [
        "mean_signal_entropy",
        "weighted_signal_entropy",
        "global_signal_entropy",
        "mean_delta_entropy",
        "weighted_delta_entropy",
        "global_delta_entropy",
    ]

    for row in dataset_rows:
        for metric in delta_metrics:
            delta_name = f"delta_{metric}_vs_original"

            if original_row is None:
                row[delta_name] = np.nan
                continue

            base_value = original_row.get(metric, np.nan)
            value = row.get(metric, np.nan)

            if np.isnan(base_value) or np.isnan(value):
                row[delta_name] = np.nan
            else:
                row[delta_name] = float(value - base_value)

    return dataset_rows


# ============================================================
# ОСНОВНОЙ ЗАПУСК
# ============================================================

all_rows = []

for dataset in DATASETS_TO_ANALYZE:
    print(f"\n=== {dataset} ===")

    dataset_rows = []

    for method in METHODS_TO_ANALYZE:
        path = SUBSET_DIR / dataset / f"{method}.pod5"

        if not path.exists():
            print(f"  {method:<24} файл не найден")
            continue

        stats = collect_entropies_from_pod5(path)

        if stats is None:
            print(f"  {method:<24} нет данных")
            continue

        row = {
            "dataset": dataset,
            "method_id": method,
            "pod5_path": str(path),
            **stats,
        }

        dataset_rows.append(row)

    dataset_rows = add_entropy_deltas_vs_original(dataset_rows)
    all_rows.extend(dataset_rows)

    # --------------------------------------------------------
    # Печать краткой таблицы по датасету
    # --------------------------------------------------------
    print()
    print(
        f"  {'Метод':<24} "
        f"{'Hsig_w':>9} "
        f"{'ΔHsig_w':>10} "
        f"{'Hsig_g':>9} "
        f"{'Hdel_w':>9} "
        f"{'ΔHdel_w':>10} "
        f"{'Hdel_g':>9}"
    )
    print(f"  {'-' * 86}")

    for row in dataset_rows:
        print(
            f"  {row['method_id']:<24} "
            f"{row['weighted_signal_entropy']:>9.3f} "
            f"{row['delta_weighted_signal_entropy_vs_original']:>+10.3f} "
            f"{row['global_signal_entropy']:>9.3f} "
            f"{row['weighted_delta_entropy']:>9.3f} "
            f"{row['delta_weighted_delta_entropy_vs_original']:>+10.3f} "
            f"{row['global_delta_entropy']:>9.3f}"
        )


# ============================================================
# СОХРАНЕНИЕ CSV
# ============================================================

df_entropy_stats = pd.DataFrame(all_rows)

out_csv = OUT_DIR / "downstream_entropy_stats.csv"
df_entropy_stats.to_csv(out_csv, index=False)

print()
print("Готово. Результаты сохранены:")
print(out_csv)

In [ ]:
import os
from pathlib import Path

# ============================================================
# ЗАПУСК DORADO ДЛЯ DATASETS, КОТОРЫЕ ОБРАБАТЫВАЮТСЯ DORADO
# ============================================================

os.environ["PATH"] += ":/content/dorado-0.9.1-linux-x64/bin"

SUBSET_DIR = Path("/content/POD5/try2/downstream_subset_new")
BC_DIR = Path("/content/POD5/try2/downstream_basecalling_new")
BC_DIR.mkdir(parents=True, exist_ok=True)

DORADO_DATASETS = {
    "Plasmid_R10_4_1": "dna_r10.4.1_e8.2_400bps_hac@v4.3.0",
    "E_coli_R9":       "dna_r9.4.1_e8_hac@v3.3",
}

METHODS_TO_BASECALL = METHOD_NAMES_FOR_POD5

OVERWRITE_FASTQ = False
MIN_READS_OK = 1


def count_fastq_reads(path: Path) -> int:
    """
    Считает число записей FASTQ.
    Нормальный FASTQ имеет 4 строки на рид.
    """
    if not path.exists():
        return 0

    try:
        result = !awk 'NR%4==1' {path} | wc -l
        return int(result[0].strip())
    except Exception:
        return 0


for dataset, model in DORADO_DATASETS.items():
    print(f"\n=== {dataset} ===")

    out_dataset_dir = BC_DIR / dataset
    out_dataset_dir.mkdir(parents=True, exist_ok=True)

    for method in METHODS_TO_BASECALL:
        pod5_path = SUBSET_DIR / dataset / f"{method}.pod5"
        out_path = out_dataset_dir / f"{method}.fastq"

        if not pod5_path.exists():
            print(f"  {method:<24} POD5 не найден, пропускаем")
            continue

        if out_path.exists() and not OVERWRITE_FASTQ:
            n_reads = count_fastq_reads(out_path)

            if n_reads >= MIN_READS_OK:
                print(f"  {method:<24} уже есть ({n_reads} ридов)")
                continue

            print(f"  {method:<24} FASTQ есть, но пустой/битый ({n_reads} ридов), пересоздаём")
            out_path.unlink()

        if out_path.exists() and OVERWRITE_FASTQ:
            out_path.unlink()

        print(f"  {method:<24}", end=" ", flush=True)

        !dorado basecaller --min-qscore 0 --emit-fastq {model} {pod5_path} > {out_path}

        n_reads = count_fastq_reads(out_path)

        if n_reads == 0:
            print("0 ридов - Dorado завершился с ошибкой")
        else:
            print(f"{n_reads} ридов")

print("\nDorado готово")

In [ ]:
import os
from pathlib import Path

# ============================================================
# ЗАПУСК GUPPY ДЛЯ Bacterial_R10_4
# ============================================================

SUBSET_DIR = Path("/content/POD5/try2/downstream_subset_new")
BC_DIR = Path("/content/POD5/try2/downstream_basecalling_new")
BC_DIR.mkdir(parents=True, exist_ok=True)

DATASET = "Bacterial_R10_4"
GUPPY_CONFIG = "dna_r10.4_e8.1_hac.cfg"

METHODS_TO_BASECALL = METHOD_NAMES_FOR_POD5

OVERWRITE_FASTQ = True

out_dataset_dir = BC_DIR / DATASET
out_dataset_dir.mkdir(parents=True, exist_ok=True)

print(f"=== {DATASET} ===")

for method in METHODS_TO_BASECALL:
    pod5_path = SUBSET_DIR / DATASET / f"{method}.pod5"
    out_fastq = out_dataset_dir / f"{method}.fastq"

    if not pod5_path.exists():
        print(f"  {method:<24} POD5 не найден, пропускаем")
        continue

    if out_fastq.exists() and not OVERWRITE_FASTQ:
        count = !awk 'NR%4==1' {out_fastq} | wc -l
        print(f"  {method:<24} уже есть ({count[0].strip()} ридов)")
        continue

    if out_fastq.exists() and OVERWRITE_FASTQ:
        out_fastq.unlink()

    tmp_dir = Path(f"/content/tmp_guppy_{DATASET}_{method}")
    tmp_dir.mkdir(parents=True, exist_ok=True)

    dst = tmp_dir / f"{method}.pod5"

    if dst.exists() or dst.is_symlink():
        dst.unlink()

    os.symlink(str(pod5_path), str(dst))

    guppy_out = Path(f"/content/guppy_{DATASET}_{method}")
    guppy_out.mkdir(parents=True, exist_ok=True)

    print(f"  {method:<24}", end=" ", flush=True)

    !/content/ont-guppy/bin/guppy_basecaller \
        --input_path {tmp_dir} \
        --save_path {guppy_out} \
        --config {GUPPY_CONFIG} \
        --device cuda:0 \
        --min_qscore 0 \
        --records_per_fastq 0

    # Собираем и pass, и fail, если они есть.
    !cat {guppy_out}/pass/*.fastq {guppy_out}/fail/*.fastq 2>/dev/null > {out_fastq}

    count = !awk 'NR%4==1' {out_fastq} | wc -l
    print(f"{count[0].strip()} ридов")

print("\nGuppy готово")

In [ ]:
import edlib
import numpy as np
from pathlib import Path

# ============================================================
# СРАВНЕНИЕ FASTQ С original.fastq
# ============================================================

BC_DIR = Path("/content/POD5/try2/downstream_basecalling_new")

DATASETS_TO_COMPARE = [
    "Plasmid_R10_4_1",
    "E_coli_R9",
    "Bacterial_R10_4",
]

# Берём тот же список, который использовался для создания POD5/basecalling
METHODS_TO_COMPARE = [
    m for m in METHOD_NAMES_FOR_POD5
    if m != "original"
]


def parse_fastq(path):
    """
    Читает FASTQ и возвращает словарь:
        read_id -> sequence
    """
    reads = {}

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        while True:
            header = f.readline().strip()
            if not header:
                break

            seq = f.readline().strip()
            f.readline()  # +
            f.readline()  # quality

            read_id = header.split()[0][1:]
            reads[read_id] = seq.upper()

    return reads


def pair_identity(seq1, seq2):
    """
    Identity = 1 - edit_distance / max(len(seq1), len(seq2))
    """
    if not seq1 or not seq2:
        denom = max(len(seq1), len(seq2))
        return 0.0, denom, denom

    dist = edlib.align(seq1, seq2, mode="NW", task="distance")["editDistance"]
    denom = max(len(seq1), len(seq2))

    if denom == 0:
        return 0.0, dist, denom

    return 1.0 - dist / denom, dist, denom


def compare_to_original(original_reads, method_reads):
    common = sorted(set(original_reads) & set(method_reads))

    if not common:
        return {
            "common": 0,
            "common_pct_vs_original": 0.0,
            "common_pct_vs_method": 0.0,
            "mean_identity": np.nan,
            "median_identity": np.nan,
            "weighted_identity": np.nan,
            "min_identity": np.nan,
        }

    identities = []
    total_dist = 0
    total_denom = 0

    for rid in common:
        identity, dist, denom = pair_identity(
            original_reads[rid],
            method_reads[rid]
        )

        identities.append(identity)
        total_dist += dist
        total_denom += denom

    identities = np.array(identities, dtype=np.float64)

    weighted_identity = (
        1.0 - total_dist / total_denom
        if total_denom > 0
        else np.nan
    )

    return {
        "common": len(common),
        "common_pct_vs_original": len(common) / len(original_reads) * 100 if original_reads else 0.0,
        "common_pct_vs_method": len(common) / len(method_reads) * 100 if method_reads else 0.0,
        "mean_identity": float(np.mean(identities)),
        "median_identity": float(np.median(identities)),
        "weighted_identity": float(weighted_identity),
        "min_identity": float(np.min(identities)),
    }


results = []

for dataset in DATASETS_TO_COMPARE:
    print(f"\n=== {dataset} ===")

    orig_path = BC_DIR / dataset / "original.fastq"

    if not orig_path.exists():
        print(f"  original.fastq не найден: {orig_path}")
        continue

    original_reads = parse_fastq(orig_path)

    print(f"  original: {len(original_reads)} ридов")
    print()
    print(
        f"  {'Метод':<24} "
        f"{'FASTQ':>8} "
        f"{'Общих':>8} "
        f"{'Общих/orig':>11} "
        f"{'Mean':>9} "
        f"{'Median':>9} "
        f"{'Weighted':>10} "
        f"{'Min':>8}"
    )
    print(f"  {'-' * 95}")

    for method in METHODS_TO_COMPARE:
        method_path = BC_DIR / dataset / f"{method}.fastq"

        if not method_path.exists():
            print(f"  {method:<24} файл не найден")
            continue

        method_reads = parse_fastq(method_path)
        stats = compare_to_original(original_reads, method_reads)

        row = {
            "dataset": dataset,
            "method_id": method,
            "original_reads": len(original_reads),
            "method_reads": len(method_reads),
            **stats,
        }

        results.append(row)

        print(
            f"  {method:<24} "
            f"{len(method_reads):>8} "
            f"{stats['common']:>8} "
            f"{stats['common_pct_vs_original']:>10.2f}% "
            f"{stats['mean_identity']*100:>8.2f}% "
            f"{stats['median_identity']*100:>8.2f}% "
            f"{stats['weighted_identity']*100:>9.2f}% "
            f"{stats['min_identity']*100:>7.2f}%"
        )

df_fastq_compare = pd.DataFrame(results)

out_csv = Path("/content/POD5/try2/downstream_fastq_identity_vs_original.csv")
df_fastq_compare.to_csv(out_csv, index=False)

print(f"\nГотово. Результаты сохранены:")
print(out_csv)

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# РАЗМЕРЫ POD5-ФАЙЛОВ И CR_POD5
# ============================================================

SUBSET_DIR = Path("/content/POD5/try2/downstream_subset_new")

DATASETS_TO_CHECK = [
    "Plasmid_R10_4_1",
    "E_coli_R9",
    "Bacterial_R10_4",
]

METHODS_TO_CHECK = METHOD_NAMES_FOR_POD5

rows = []

for ds in DATASETS_TO_CHECK:
    print(f"\n=== {ds} ===")

    orig_path = SUBSET_DIR / ds / "original.pod5"

    if not orig_path.exists():
        print(f"  original.pod5 не найден: {orig_path}")
        continue

    orig_size = orig_path.stat().st_size

    print(f"  {'Метод':<24} {'МБ':>10} {'CR_POD5':>10}")
    print(f"  {'-' * 48}")

    for method in METHODS_TO_CHECK:
        f = SUBSET_DIR / ds / f"{method}.pod5"

        if not f.exists():
            print(f"  {method:<24} файл не найден")
            continue

        size = f.stat().st_size
        cr = orig_size / size if size > 0 else float("nan")

        rows.append({
            "dataset": ds,
            "method_id": method,
            "pod5_size_bytes": size,
            "pod5_size_mb": size / 1e6,
            "original_size_bytes": orig_size,
            "cr_pod5": cr,
        })

        print(f"  {method:<24} {size / 1e6:>9.2f} {cr:>9.3f}x")

df_pod5_sizes = pd.DataFrame(rows)

out_csv = Path("/content/POD5/try2/downstream_pod5_size_cr.csv")
df_pod5_sizes.to_csv(out_csv, index=False)

print("\nГотово. Результаты сохранены:")
print(out_csv)

### --- Модельные результаты бейскорлов (из сетов для сравнения)

In [ ]:
!pip install pysam

In [ ]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import edlib
import pysam

# ============================================================
# СРАВНЕНИЕ FASTQ С ЭТАЛОННЫМИ BASECALLS ИЗ BAM
# ============================================================

BAM_PATH = Path("/content/drive/MyDrive/calls_2025-09-17_T19-07-14.bam")

BC_DIR = Path("/content/POD5/try2/downstream_basecalling_new")
OUT_DIR = Path("/content/POD5/try2")

DATASET = "E_coli_R9"

# Берём тот же список методов, который использовался при создании POD5/basecalling
METHODS_TO_COMPARE = METHOD_NAMES_FOR_POD5

# Если True, из BAM берём только primary alignments.
# Это обычно правильно, потому что secondary/supplementary могут дублировать read_id.
SKIP_SECONDARY_AND_SUPPLEMENTARY = True

# Если True, unmapped-записи из BAM тоже берём.
# Для BAM с basecalls это допустимо: даже unmapped read может содержать последовательность.
KEEP_UNMAPPED = True


# ============================================================
# ФУНКЦИИ
# ============================================================

COMP = str.maketrans(
    "ACGTUNacgtun",
    "TGCAANtgcaan"
)


def revcomp(seq: str) -> str:
    return seq.translate(COMP)[::-1]


def open_text_auto(path: Path):
    path = Path(path)
    if path.name.lower().endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", errors="replace")
    return open(path, "rt", encoding="utf-8", errors="replace")


def normalize_fastq_read_id(header: str) -> str:
    """
    FASTQ header обычно имеет вид:
        @read_id runid=... sampleid=...

    Берём только первый токен без символа @.
    """
    rid = header.strip().split()[0]
    if rid.startswith("@"):
        rid = rid[1:]
    return rid


def parse_fastq(path: Path) -> dict:
    """
    Читает FASTQ или FASTQ.GZ.

    Возвращает:
        {read_id: sequence}
    """
    reads = {}

    with open_text_auto(path) as f:
        while True:
            header = f.readline()
            if not header:
                break

            seq = f.readline().strip()
            plus = f.readline()
            qual = f.readline()

            if not qual:
                break

            rid = normalize_fastq_read_id(header)
            reads[rid] = seq.upper()

    return reads


def get_bam_sequence_in_read_orientation(rec):
    """
    Возвращает последовательность рида в исходной read-ориентации.

    В aligned BAM последовательность для рида на reverse strand может быть
    записана в reverse-complement ориентации. Поэтому нужно привести её
    к исходной ориентации рида.
    """
    if rec.query_sequence is None:
        return None

    # В новых версиях pysam есть готовый метод.
    if hasattr(rec, "get_forward_sequence"):
        seq = rec.get_forward_sequence()
        if seq is not None:
            return seq.upper()

    # Резервный вариант.
    seq = rec.query_sequence.upper()
    if rec.is_reverse:
        seq = revcomp(seq)

    return seq


def parse_bam_as_original_read_orientation(path: Path) -> dict:
    """
    Читает BAM и возвращает последовательности в исходной read-ориентации.

    Возвращает:
        {read_id: sequence}
    """
    reads = {}

    total_records = 0
    used_records = 0
    skipped_secondary = 0
    skipped_unmapped = 0
    skipped_no_seq = 0
    duplicated_ids = 0
    reverse_records = 0

    with pysam.AlignmentFile(str(path), "rb", check_sq=False) as bam:
        for rec in bam.fetch(until_eof=True):
            total_records += 1

            if SKIP_SECONDARY_AND_SUPPLEMENTARY:
                if rec.is_secondary or rec.is_supplementary:
                    skipped_secondary += 1
                    continue

            if rec.is_unmapped and not KEEP_UNMAPPED:
                skipped_unmapped += 1
                continue

            rid = rec.query_name
            seq = get_bam_sequence_in_read_orientation(rec)

            if seq is None:
                skipped_no_seq += 1
                continue

            if rec.is_reverse:
                reverse_records += 1

            if rid in reads:
                duplicated_ids += 1
                continue

            reads[rid] = seq
            used_records += 1

    print("=== BAM loaded ===")
    print(f"BAM path: {path}")
    print(f"Всего записей в BAM: {total_records}")
    print(f"Использовано записей: {used_records}")
    print(f"Уникальных read_id: {len(reads)}")
    print(f"Reverse-записей приведено к read-ориентации: {reverse_records}")
    print(f"Пропущено secondary/supplementary: {skipped_secondary}")
    print(f"Пропущено unmapped: {skipped_unmapped}")
    print(f"Пропущено без sequence: {skipped_no_seq}")
    print(f"Дубликатов read_id пропущено: {duplicated_ids}")
    print()

    return reads


def pair_identity(seq_ref: str, seq_test: str) -> tuple:
    """
    Считает identity между двумя последовательностями.

    Формула:
        identity = 1 - edit_distance / max(len(seq_ref), len(seq_test))

    Возвращает:
        identity, edit_distance, denominator
    """
    denom = max(len(seq_ref), len(seq_test))

    if denom == 0:
        return 0.0, 0, 0

    if not seq_ref or not seq_test:
        return 0.0, denom, denom

    dist = edlib.align(
        seq_ref,
        seq_test,
        mode="NW",
        task="distance"
    )["editDistance"]

    identity = 1.0 - dist / denom

    return identity, dist, denom


def compare_fastq_to_bam(method_name: str, fastq_path: Path, bam_reads: dict) -> dict:
    """
    Сравнивает FASTQ одного метода с эталонными basecalls из BAM.
    """
    method_reads = parse_fastq(fastq_path)

    common = sorted(set(bam_reads) & set(method_reads))

    if not common:
        return {
            "dataset": DATASET,
            "method_id": method_name,
            "bam_reads": len(bam_reads),
            "fastq_reads": len(method_reads),
            "common": 0,
            "common_pct_vs_bam": 0.0,
            "common_pct_vs_fastq": 0.0,
            "mean_identity": np.nan,
            "median_identity": np.nan,
            "weighted_identity": np.nan,
            "min_identity": np.nan,
            "max_identity": np.nan,
        }

    identities = []
    total_dist = 0
    total_denom = 0

    for rid in common:
        s_bam = bam_reads[rid]
        s_method = method_reads[rid]

        identity, dist, denom = pair_identity(s_bam, s_method)

        identities.append(identity)
        total_dist += dist
        total_denom += denom

    identities = np.array(identities, dtype=np.float64)

    weighted_identity = (
        1.0 - total_dist / total_denom
        if total_denom > 0
        else np.nan
    )

    return {
        "dataset": DATASET,
        "method_id": method_name,
        "bam_reads": len(bam_reads),
        "fastq_reads": len(method_reads),
        "common": len(common),
        "common_pct_vs_bam": len(common) / len(bam_reads) * 100 if bam_reads else 0.0,
        "common_pct_vs_fastq": len(common) / len(method_reads) * 100 if method_reads else 0.0,
        "mean_identity": float(np.mean(identities)),
        "median_identity": float(np.median(identities)),
        "weighted_identity": float(weighted_identity),
        "min_identity": float(np.min(identities)),
        "max_identity": float(np.max(identities)),
    }


# ============================================================
# ОСНОВНОЙ ЗАПУСК
# ============================================================

bam_reads = parse_bam_as_original_read_orientation(BAM_PATH)

print(f"=== {DATASET}: identity vs эталонные BAM-basecalls ===")
print(f"BAM ридов: {len(bam_reads)}")
print()

header = (
    f"{'Метод':<24} "
    f"{'FASTQ':>8} "
    f"{'Общих':>8} "
    f"{'Общих/BAM':>11} "
    f"{'Общих/FQ':>10} "
    f"{'Mean':>9} "
    f"{'Median':>9} "
    f"{'Weighted':>10} "
    f"{'Min':>8}"
)

print(header)
print("-" * len(header))

results = []

for method in METHODS_TO_COMPARE:
    fastq_path = BC_DIR / DATASET / f"{method}.fastq"

    if not fastq_path.exists():
        print(f"{method:<24} файл не найден: {fastq_path}")
        continue

    res = compare_fastq_to_bam(method, fastq_path, bam_reads)
    results.append(res)

    print(
        f"{res['method_id']:<24} "
        f"{res['fastq_reads']:>8} "
        f"{res['common']:>8} "
        f"{res['common_pct_vs_bam']:>10.2f}% "
        f"{res['common_pct_vs_fastq']:>9.2f}% "
        f"{res['mean_identity'] * 100:>8.2f}% "
        f"{res['median_identity'] * 100:>8.2f}% "
        f"{res['weighted_identity'] * 100:>9.2f}% "
        f"{res['min_identity'] * 100:>7.2f}%"
    )

df_bam_compare = pd.DataFrame(results)

out_csv = OUT_DIR / f"downstream_bam_identity_{DATASET}.csv"
df_bam_compare.to_csv(out_csv, index=False)

print()
print("Готово. Результаты сохранены:")
print(out_csv)

In [ ]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import edlib

# ============================================================
# СРАВНЕНИЕ FASTQ С ЭТАЛОННЫМ SUP-BASECALLING ДЛЯ PLASMID
# ============================================================

DATASET = "Plasmid_R10_4_1"

BC_DIR = Path("/content/POD5/try2/downstream_basecalling_new")
OUT_DIR = Path("/content/POD5/try2")

# Эталонный SUP-бейсколлинг для этого же набора данных
REFERENCE_FASTQ = Path("/content/drive/MyDrive/original_sup.fastq")

# Берём тот же список методов, который использовался при создании POD5/basecalling
METHODS_TO_COMPARE = METHOD_NAMES_FOR_POD5

# Сколько несовпавших read_id печатать в диагностике
N_MISSING_TO_PRINT = 20


# ============================================================
# ФУНКЦИИ
# ============================================================

def open_text_auto(path: Path):
    path = Path(path)

    if path.name.lower().endswith(".gz"):
        return gzip.open(
            path,
            "rt",
            encoding="utf-8",
            errors="replace"
        )

    return open(
        path,
        "rt",
        encoding="utf-8",
        errors="replace"
    )


def normalize_fastq_read_id(header: str) -> str:
    """
    FASTQ header обычно имеет вид:
        @read_id runid=... sampleid=...

    Берём только первый токен без символа @.
    """
    rid = header.strip().split()[0]

    if rid.startswith("@"):
        rid = rid[1:]

    return rid


def parse_fastq(path: Path, verbose: bool = False) -> dict:
    """
    Читает FASTQ или FASTQ.GZ.

    Возвращает:
        {read_id: sequence}
    """
    reads = {}
    n_records = 0
    n_duplicates = 0

    with open_text_auto(path) as f:
        while True:
            header = f.readline()

            if not header:
                break

            seq = f.readline().strip()
            plus = f.readline()
            qual = f.readline()

            if not qual:
                break

            n_records += 1
            rid = normalize_fastq_read_id(header)

            if rid in reads:
                n_duplicates += 1
                continue

            reads[rid] = seq.upper()

    if verbose:
        print(f"Loaded FASTQ: {path}")
        print(f"  records: {n_records}")
        print(f"  unique read_id: {len(reads)}")
        print(f"  duplicates skipped: {n_duplicates}")
        print()

    return reads


def pair_identity(seq_ref: str, seq_test: str) -> tuple:
    """
    Считает identity между двумя последовательностями.

    Формула:
        identity = 1 - edit_distance / max(len(seq_ref), len(seq_test))

    Возвращает:
        identity, edit_distance, denominator
    """
    denom = max(len(seq_ref), len(seq_test))

    if denom == 0:
        return 0.0, 0, 0

    if not seq_ref or not seq_test:
        return 0.0, denom, denom

    dist = edlib.align(
        seq_ref,
        seq_test,
        mode="NW",
        task="distance"
    )["editDistance"]

    identity = 1.0 - dist / denom

    return identity, dist, denom


def compare_fastq_to_reference(
    method_name: str,
    method_fastq: Path,
    reference_reads: dict
) -> dict:
    """
    Сравнивает FASTQ одного метода с эталонным FASTQ.
    """
    method_reads = parse_fastq(method_fastq)

    common = sorted(set(reference_reads) & set(method_reads))

    missing_from_reference = sorted(set(method_reads) - set(reference_reads))
    missing_from_method = sorted(set(reference_reads) - set(method_reads))

    if not common:
        return {
            "dataset": DATASET,
            "method_id": method_name,
            "reference_reads": len(reference_reads),
            "method_reads": len(method_reads),
            "common": 0,
            "common_pct_vs_reference": 0.0,
            "common_pct_vs_method": 0.0,
            "mean_identity": np.nan,
            "median_identity": np.nan,
            "weighted_identity": np.nan,
            "min_identity": np.nan,
            "max_identity": np.nan,
            "p1_identity": np.nan,
            "p5_identity": np.nan,
            "missing_from_reference_count": len(missing_from_reference),
            "missing_from_method_count": len(missing_from_method),
            "_missing_from_reference": missing_from_reference,
            "_missing_from_method": missing_from_method,
        }

    identities = []
    total_dist = 0
    total_denom = 0

    for rid in common:
        s_ref = reference_reads[rid]
        s_method = method_reads[rid]

        identity, dist, denom = pair_identity(s_ref, s_method)

        identities.append(identity)
        total_dist += dist
        total_denom += denom

    identities = np.array(identities, dtype=np.float64)

    weighted_identity = (
        1.0 - total_dist / total_denom
        if total_denom > 0
        else np.nan
    )

    return {
        "dataset": DATASET,
        "method_id": method_name,
        "reference_reads": len(reference_reads),
        "method_reads": len(method_reads),
        "common": len(common),
        "common_pct_vs_reference": len(common) / len(reference_reads) * 100 if reference_reads else 0.0,
        "common_pct_vs_method": len(common) / len(method_reads) * 100 if method_reads else 0.0,
        "mean_identity": float(np.mean(identities)),
        "median_identity": float(np.median(identities)),
        "weighted_identity": float(weighted_identity),
        "min_identity": float(np.min(identities)),
        "max_identity": float(np.max(identities)),
        "p1_identity": float(np.percentile(identities, 1)),
        "p5_identity": float(np.percentile(identities, 5)),
        "missing_from_reference_count": len(missing_from_reference),
        "missing_from_method_count": len(missing_from_method),
        "_missing_from_reference": missing_from_reference,
        "_missing_from_method": missing_from_method,
    }


# ============================================================
# ПРОВЕРКИ ПУТЕЙ
# ============================================================

if not REFERENCE_FASTQ.exists():
    raise FileNotFoundError(f"Не найден reference FASTQ: {REFERENCE_FASTQ}")

dataset_dir = BC_DIR / DATASET

if not dataset_dir.exists():
    raise FileNotFoundError(f"Не найдена папка с методами: {dataset_dir}")


# ============================================================
# ОСНОВНОЙ ЗАПУСК
# ============================================================

print(f"=== {DATASET}: identity vs эталонный SUP FASTQ ===")
print(f"Reference FASTQ: {REFERENCE_FASTQ}")
print()

reference_reads = parse_fastq(REFERENCE_FASTQ, verbose=True)

print(f"Reference FASTQ reads: {len(reference_reads)}")
print()

results = []

for method in METHODS_TO_COMPARE:
    method_fastq = dataset_dir / f"{method}.fastq"

    if not method_fastq.exists():
        print(f"{method:<24} файл не найден: {method_fastq}")
        continue

    res = compare_fastq_to_reference(
        method_name=method,
        method_fastq=method_fastq,
        reference_reads=reference_reads
    )

    results.append(res)


# ============================================================
# РАСЧЁТ DELTA ОТНОСИТЕЛЬНО ORIGINAL
# ============================================================

baseline_mean = None
baseline_weighted = None

for r in results:
    if r["method_id"] == "original":
        baseline_mean = r["mean_identity"]
        baseline_weighted = r["weighted_identity"]
        break

for r in results:
    if baseline_mean is not None and not np.isnan(r["mean_identity"]):
        r["delta_mean_vs_original_pp"] = (
            r["mean_identity"] - baseline_mean
        ) * 100
    else:
        r["delta_mean_vs_original_pp"] = np.nan

    if baseline_weighted is not None and not np.isnan(r["weighted_identity"]):
        r["delta_weighted_vs_original_pp"] = (
            r["weighted_identity"] - baseline_weighted
        ) * 100
    else:
        r["delta_weighted_vs_original_pp"] = np.nan


# ============================================================
# ВЫВОД ТАБЛИЦЫ
# ============================================================

print()
print(f"=== {DATASET}: summary vs SUP reference ===")

header = (
    f"{'Метод':<24} "
    f"{'FASTQ':>8} "
    f"{'Общих':>8} "
    f"{'Общих/ref':>11} "
    f"{'Общих/FQ':>10} "
    f"{'Mean':>9} "
    f"{'ΔMean':>9} "
    f"{'Median':>9} "
    f"{'Weighted':>10} "
    f"{'ΔWeighted':>11} "
    f"{'P5':>8} "
    f"{'Min':>8}"
)

print(header)
print("-" * len(header))

for r in results:
    print(
        f"{r['method_id']:<24} "
        f"{r['method_reads']:>8} "
        f"{r['common']:>8} "
        f"{r['common_pct_vs_reference']:>10.2f}% "
        f"{r['common_pct_vs_method']:>9.2f}% "
        f"{r['mean_identity'] * 100:>8.2f}% "
        f"{r['delta_mean_vs_original_pp']:>+8.2f} "
        f"{r['median_identity'] * 100:>8.2f}% "
        f"{r['weighted_identity'] * 100:>9.2f}% "
        f"{r['delta_weighted_vs_original_pp']:>+10.2f} "
        f"{r['p5_identity'] * 100:>7.2f}% "
        f"{r['min_identity'] * 100:>7.2f}%"
    )


# ============================================================
# СОХРАНЕНИЕ CSV
# ============================================================

# В CSV не сохраняем сами списки read_id, только их количества.
rows_for_csv = []

for r in results:
    row = {
        k: v
        for k, v in r.items()
        if not k.startswith("_")
    }
    rows_for_csv.append(row)

df_sup_compare = pd.DataFrame(rows_for_csv)

out_csv = OUT_DIR / f"downstream_sup_identity_{DATASET}.csv"
df_sup_compare.to_csv(out_csv, index=False)

print()
print("Готово. Результаты сохранены:")
print(out_csv)


# ============================================================
# ДИАГНОСТИКА НЕСОВПАВШИХ READ_ID ДЛЯ ORIGINAL
# ============================================================

for r in results:
    if r["method_id"] == "original":
        print()
        print("=== Диагностика read_id для original ===")
        print(f"Reference reads: {r['reference_reads']}")
        print(f"Method reads:    {r['method_reads']}")
        print(f"Common:          {r['common']}")
        print(f"Нет в reference: {r['missing_from_reference_count']}")
        print(f"Нет в method:    {r['missing_from_method_count']}")

        if r["_missing_from_reference"]:
            print()
            print("Первые read_id, которые есть в method original, но нет в reference:")
            for rid in r["_missing_from_reference"][:N_MISSING_TO_PRINT]:
                print(" ", rid)

        if r["_missing_from_method"]:
            print()
            print("Первые read_id, которые есть в reference, но нет в method original:")
            for rid in r["_missing_from_method"][:N_MISSING_TO_PRINT]:
                print(" ", rid)

        break

# Вспомогательные блоки

### Копируем на диск

In [ ]:
SRC = Path("/content/POD5/try2/downstream_subset_new")
DST = Path("/content/drive/MyDrive/pod5_try2/downstream_subset_new")

print("Копируем на диск...")
shutil.copytree(str(SRC), str(DST), dirs_exist_ok=True)

# Проверяем
for ds in ["Plasmid_R10_4_1", "E_coli_R9", "Bacterial_R10_4"]:
    files = list((DST / ds).glob("*.pod5"))
    print(f"  {ds}: {len(files)} файлов")

print("Готово")

In [ ]:
import shutil
from pathlib import Path

SRC = Path("/content/drive/MyDrive/pod5_try2/downstream_subset")
DST = Path("/content/POD5/try2/downstream_subset")
DST.mkdir(parents=True, exist_ok=True)

shutil.copytree(str(SRC), str(DST), dirs_exist_ok=True)

# Проверяем
for dataset in ["Plasmid_R10_4_1", "E_coli_R9", "Bacterial_R10_4"]:
    f = DST / dataset / "original.pod5"
    print(f"{dataset}: {'v' if f.exists() else 'x'}")

In [ ]:
shutil.copytree(
    "/content/POD5/try2",
    "/content/drive/MyDrive/pod5_try2",
    dirs_exist_ok=True
)
print("Готово")

### Копируем с диска

In [ ]:
SRC = Path("/content/drive/MyDrive/POD5/try2/downstream_subset_new")
DST = Path("/content/POD5/try2/downstream_subset_new")

DST.mkdir(parents=True, exist_ok=True)

print("Копируем с диска...")
shutil.copytree(str(SRC), str(DST), dirs_exist_ok=True)

# Проверяем
for ds in ["Plasmid_R10_4_1", "E_coli_R9", "Bacterial_R10_4"]:
    files = list((DST / ds).glob("*.pod5"))
    print(f"  {ds}: {len(files)} файлов")

print("отово")

In [ ]:
from pathlib import Path
import pandas as pd
import re

# ------------------------------------------------------------
# 1. Корневая папка
# ------------------------------------------------------------

BASE_DIR = Path("/content/POD5")

# Если точно знаешь папку с созданными downstream POD5, можно указать её здесь.
# Например:
# SEARCH_DIR = Path("/content/POD5/try2/downstream_subset_new")
# Если не уверен — оставь BASE_DIR, будет искать рекурсивно по всему /content/POD5.
SEARCH_DIR = BASE_DIR

print("Ищу в:", SEARCH_DIR)
print("Папка существует:", SEARCH_DIR.exists())


# ------------------------------------------------------------
# 2. Актуальный список методов
# ------------------------------------------------------------

METHOD_GROUPS = {
    "amplitude_quantization": [
        "lsb_k1", "lsb_k2", "lsb_k3",
        "quant_q2", "quant_q4", "quant_q8", "quant_q16",
        "bound_eps1", "bound_eps2", "bound_eps4", "bound_eps8",
    ],

    "predictive_coding": [
        "dpcm_q1", "dpcm_q2", "dpcm_q4", "dpcm_q8",
    ],

    "nonlinear_companding": [
        "mulaw_50", "mulaw_100", "mulaw_255",
        "mulaw_q128",  # оставил, потому что в твоём логе было именно mulaw_q128
    ],

    "run_length_after_quantization": [
        "rle_quant_q2", "rle_quant_q4", "rle_quant_q8", "rle_quant_q16",
    ],

    "temporal_resolution_reduction": [
        "paa_w2", "paa_w4", "paa_w8",
        "downsample_d2", "downsample_d4",
    ],

    "piecewise_approximation": [
        "pla_eps1", "pla_eps2", "pla_eps4", "pla_eps8", "pla_eps16",
    ],

    "frequency_domain": [
        "fft_50", "fft_25", "fft_10",
    ],
}

METHOD_TO_GROUP = {
    method: group
    for group, methods in METHOD_GROUPS.items()
    for method in methods
}

ALL_METHODS = list(METHOD_TO_GROUP.keys())


# ------------------------------------------------------------
# 3. Определение датасета по пути
# ------------------------------------------------------------

DATASET_PATTERNS = {
    "Plasmid_R10_4_1": [
        "plasmid", "plasmid_r10_4_1", "r10.4.1", "r10_4_1"
    ],
    "E_coli_R9": [
        "e_coli_r9", "e coli", "e.coli", "err9127551", "r9 data", "r9"
    ],
    "Bacterial_R10_4": [
        "bacterial", "bacterial_r10_4", "r10.4 data", "r10_4"
    ],
}

def detect_dataset(path: Path) -> str | None:
    s = str(path).lower()

    # Более специфичные проверки сначала
    if "plasmid" in s or "r10_4_1" in s or "r10.4.1" in s:
        return "Plasmid_R10_4_1"

    if "err9127551" in s or "e_coli_r9" in s or "e coli" in s or "r9 data" in s:
        return "E_coli_R9"

    if "bacterial" in s or "bacterial_r10_4" in s or "r10.4 data" in s:
        return "Bacterial_R10_4"

    return None


# ------------------------------------------------------------
# 4. Определение метода по имени файла или пути
# ------------------------------------------------------------

def detect_method(path: Path) -> str | None:
    s = str(path).lower()

    # Ищем самые длинные method_id первыми, чтобы не спутать, например q8 и q16
    for method in sorted(ALL_METHODS, key=len, reverse=True):
        if method.lower() in s:
            return method

    return None


# ------------------------------------------------------------
# 5. Сканирование существующих POD5-файлов
# ------------------------------------------------------------

pod5_files = sorted(SEARCH_DIR.rglob("*.pod5"))

print(f"Найдено POD5-файлов: {len(pod5_files)}")

rows = []

for fp in pod5_files:
    dataset = detect_dataset(fp)
    method = detect_method(fp)

    # Пропускаем оригинальные POD5 без method_id
    if method is None:
        continue

    rows.append({
        "dataset": dataset if dataset is not None else "UNKNOWN",
        "group": METHOD_TO_GROUP.get(method, "UNKNOWN"),
        "method_id": method,
        "filename": fp.name,
        "path": str(fp),
        "size_mb": fp.stat().st_size / 1_000_000,
    })

df_files = pd.DataFrame(rows)

print(f"Найдено файлов с распознанными method_id: {len(df_files)}")

if df_files.empty:
    print("Не найдено файлов с method_id. Проверь SEARCH_DIR или имена файлов.")
else:
    display(df_files.sort_values(["dataset", "group", "method_id", "filename"]))


# ------------------------------------------------------------
# 6. Таблица наличия методов по датасетам
# ------------------------------------------------------------

EXPECTED_DATASETS = ["Plasmid_R10_4_1", "E_coli_R9", "Bacterial_R10_4"]

if not df_files.empty:
    # Если для одного method_id/dataset найдено несколько файлов, считаем количество
    count_table = (
        df_files
        .groupby(["method_id", "dataset"])
        .size()
        .unstack(fill_value=0)
    )

    # Добавляем отсутствующие датасеты, если их вообще нет в результатах
    for ds in EXPECTED_DATASETS:
        if ds not in count_table.columns:
            count_table[ds] = 0

    count_table = count_table[EXPECTED_DATASETS]

    # Добавляем методы, которых пока нет вообще
    for method in ALL_METHODS:
        if method not in count_table.index:
            count_table.loc[method] = 0

    count_table = count_table.loc[ALL_METHODS]

    status_table = count_table.map(lambda x: "✓" if x > 0 else "—")

    status_table.insert(0, "group", [METHOD_TO_GROUP[m] for m in status_table.index])

    print("\nМатрица наличия файлов:")
    display(status_table)

    print("\nКоличество файлов на метод/датасет:")
    count_table_with_group = count_table.copy()
    count_table_with_group.insert(0, "group", [METHOD_TO_GROUP[m] for m in count_table_with_group.index])
    display(count_table_with_group)


# ------------------------------------------------------------
# 7. Что есть не для всех трёх датасетов
# ------------------------------------------------------------

if not df_files.empty:
    missing_rows = []

    for method in ALL_METHODS:
        existing = set(df_files.loc[df_files["method_id"] == method, "dataset"])
        existing = {x for x in existing if x != "UNKNOWN"}

        missing = [ds for ds in EXPECTED_DATASETS if ds not in existing]

        if missing and existing:
            missing_rows.append({
                "group": METHOD_TO_GROUP[method],
                "method_id": method,
                "exists_for": ", ".join(sorted(existing)),
                "missing_for": ", ".join(missing),
            })

    df_missing = pd.DataFrame(missing_rows)

    print("\nМетоды, которые есть не для всех датасетов:")
    if df_missing.empty:
        print("Все найденные методы либо есть для всех трёх датасетов, либо не найдены вообще.")
    else:
        display(df_missing)


# ------------------------------------------------------------
# 8. Методы, которых нет вообще
# ------------------------------------------------------------

if not df_files.empty:
    existing_methods = set(df_files["method_id"])
    absent_methods = [m for m in ALL_METHODS if m not in existing_methods]

    df_absent = pd.DataFrame({
        "group": [METHOD_TO_GROUP[m] for m in absent_methods],
        "method_id": absent_methods,
    })

    print("\nМетоды, файлов для которых не найдено вообще:")
    if df_absent.empty:
        print("Нет таких методов.")
    else:
        display(df_absent)

In [ ]:
!mkdir -p /content/POD5/try2
!cp -r /content/drive/MyDrive/POD5/try2/downstream_subset_new /content/POD5/try2/